# 🔁 20 · Hidden Markov models II: learning, Baum–Welch and beyond
### Counting when the states are known, EM when they are hidden, choosing the number of states — and the HMM family used in genomics: semi-Markov durations, profile HMMs and phylo-HMMs

**BiologyToML** · Level 1 · 02-620 core · Part D — Probabilistic graphical models · Module 20 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/20_hmm_learning.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib |
| **Prerequisites** | Module 19 (forward, backward, posterior decoding, scaling — this notebook re-states what it needs), Module 13 (mixture models and the EM algorithm), Module 05 (maximum likelihood, pseudo-counts/MAP). |
| **Estimated time** | 175 min (reading + running + the ✏️ exercises) |
| **02-620 topics covered** | Hidden Markov models (part 2: learning) and dynamic Bayesian networks: learning with observed states (maximum likelihood = counting; the three-sequence two-dice example), learning with hidden states (the EM or Baum–Welch algorithm: expected counts $S_t(i)$ and $S_t(i,j)$ from the forward–backward variables, M-step by counting), choosing the number of states (model selection), dynamic Bayesian networks (factorial, mixture-of-Gaussians, autoregressive, input–output, semi-Markov and hierarchical HMMs), DNA motifs, profile HMMs for sequence alignment and phylo-HMMs for conserved elements |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> The two-dice model, the three labelled example sequences and the five-sequence DNA alignment of the profile HMM are worked examples of the 02-620 lecture, typed in. All other dice rolls, durations and the three-species phylo-HMM alignment (cells 7.4–7.5) are **SIMULATED**. Every number quoted in the text is printed by the code.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Derive** the maximum-likelihood estimates of an HMM with observed states — initial, transition and emission probabilities are count ratios — and **compute** every number of the classic three-sequence example.
2. **Explain** why hidden states turn learning into a chicken-and-egg problem and how the EM algorithm resolves it with *expected* counts, exactly as for mixture models.
3. **Derive** $S_t(i)=P(q_t=s_i\mid O)$ and $S_t(i,j)=P(q_t=s_i,q_{t+1}=s_j\mid O)$ from the forward–backward variables, and **verify** them by enumerating all paths.
4. **Implement** Baum–Welch from scratch (scaled, for many sequences) and **interpret** its behaviour: a log-likelihood that never decreases, label switching, local optima and plateaus, and the amount of data needed.
5. **Choose** the number of hidden states with held-out likelihood and BIC, and **explain** why the training likelihood cannot do it.
6. **Describe** HMMs as dynamic Bayesian networks (factorial, mixture-of-Gaussians, autoregressive, input–output, semi-Markov, hierarchical) and **compare** geometric with explicit state durations by simulation.
7. **Build** a profile HMM from a five-sequence DNA alignment, draw its architecture and score sequences with it, and **decode** conserved elements with a toy phylo-HMM.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, the two-dice HMM | 5 min |
| 1 · Learning with observed states | MLE = counting; three labelled example sequences; pseudo-counts | 25 min |
| 2 · When the states are hidden | the chicken-and-egg problem; EM; HMMs vs mixture models | 10 min |
| 3 · E-step and M-step | expected counts from forward–backward; brute-force check; M-step = counting | 25 min |
| 4 · Baum–Welch from scratch | learn the dice from rolls; label switching; restarts; how much data | 30 min |
| 5 · How many hidden states? | training vs held-out likelihood; BIC | 15 min |
| 6 · Dynamic Bayesian networks | factorial, MoG, AR, IO HMMs unrolled; semi-Markov durations; hierarchical HMMs | 25 min |
| 7 · HMMs in sequence biology | motifs, a profile HMM from a 5-sequence alignment, a toy phylo-HMM | 30 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 10 min |

## 🧫 The biological question

In Module 19 somebody *gave* us the HMM's parameters: how often the croupier switches dice, how each die is loaded.
In biology nobody hands you those numbers. What fraction of a genome is exon, and how long is a typical intron? Which
amino acids are tolerated at position 37 of a protein family? How quickly do conserved and fast-evolving regions alternate
along a chromosome? These are exactly the transition and emission probabilities of an HMM, and they must be **learned
from data** — often from data in which the hidden labels were never observed.

Learning is what turned HMMs into workhorses of computational biology. The Pfam database represents each of its roughly
twenty thousand protein families by a **profile HMM** estimated from a multiple alignment of family members (Krogh *et
al.* 1994; Eddy 1998). Gene finders are trained on curated genes; **phylo-HMMs** such as phastCons learn how conserved
elements alternate with neutral DNA across species (Siepel *et al.* 2005); ChromHMM learns chromatin states from histone
marks with no labels at all (Ernst & Kellis 2012). This notebook covers the two learning situations of the 02-620
lecture — states observed (count!) and states hidden (EM, the Baum–Welch algorithm) — then the model-selection
question "how many states?", the wider family of dynamic Bayesian networks, and two biological HMMs of this lecture
topic — profile HMMs for sequence families and phylo-HMMs for conserved elements.

> 🧪 **Simulated data, with a known truth.** The course stores no data sets: apart from small worked examples typed in
> from the lecture topic, every sequence here is generated by code from a model whose parameters we know. So we can
> measure exactly how well Baum–Welch recovers the parameters, how many states model selection picks, and how
> accurately a phylo-HMM finds conserved elements — questions that real data cannot answer, because there the truth is
> unknown.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import itertools
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`DICE_HMM` holds the two-dice model of Module 19, `accuracy` scores a decoding against the truth, and `draw_graph` draws the dynamic Bayesian networks of Section 6. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import os
import numpy as np
import matplotlib.pyplot as plt

DICE_HMM = {
    "states": ("A", "B"),
    "faces": (1, 2, 3, 4, 5, 6),
    "pi": np.array([0.7, 0.3]),                              # initial state probabilities
    "A": np.array([[0.8, 0.2],                               # A[i, j] = P(q_t = j | q_{t-1} = i)
                   [0.2, 0.8]]),
    "B": np.array([[0.3, 0.2, 0.2, 0.1, 0.1, 0.1],           # B[i, k] = P(o_t = face k+1 | q_t = i)
                   [0.1, 0.1, 0.1, 0.2, 0.2, 0.3]]),
}


def accuracy(y_true, y_pred) -> float:
    """Fraction of correct predictions."""
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def draw_graph(edges, pos: dict, directed: bool = True, ax=None, highlight=None, node_color: str = "#9ecae1",
               highlight_color: str = "#fdae6b", radius: float = 0.07, title: str = "", edge_labels=None):
    """Draw a small graph with Matplotlib only.

    ``edges`` is a list of (u, v) pairs, ``pos`` maps every node to (x, y) in [0, 1]².
    ``highlight`` is a set of nodes to colour differently (e.g. observed evidence).
    """
    from matplotlib.patches import Circle, FancyArrowPatch
    ax = ax or plt.gca()
    highlight = set(highlight or [])
    for i, (u, v) in enumerate(edges):
        (x0, y0), (x1, y1) = pos[u], pos[v]
        d = np.hypot(x1 - x0, y1 - y0) or 1.0
        sx, sy = x0 + (x1 - x0) * radius / d, y0 + (y1 - y0) * radius / d
        ex, ey = x1 - (x1 - x0) * radius / d, y1 - (y1 - y0) * radius / d
        ax.add_patch(FancyArrowPatch((sx, sy), (ex, ey), arrowstyle="-|>" if directed else "-",
                                     mutation_scale=12, lw=1.2, color="0.25"))
        if edge_labels is not None:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, str(edge_labels[i]), fontsize=7, color="0.2",
                    ha="center", va="center", bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    for node, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), radius, fc=highlight_color if node in highlight else node_color,
                            ec="0.2", lw=1, zorder=3))
        ax.text(x, y, str(node), ha="center", va="center", fontsize=9, zorder=4)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['accuracy', 'draw_graph']))

### Notation (the same as in Module 19)

$n$ hidden states $s_1..s_n$, $m$ output symbols, sequences of length $T$, and — new here — $K$ independent training
sequences $O^1, \dots, O^K$. Parameters: $\pi_i = P(q_1 = s_i)$; $a_{ij} = P(q_t = s_j \mid q_{t-1} = s_i)$
(row = *from*, column = *to*; every row sums to 1); $b_i(\sigma) = P(o_t = \sigma \mid q_t = s_i)$. In code: `pi`, `trans`,
`emit`; dice faces 1–6 are stored as indices 0–5, and Python position 0 is the time $t = 1$ of the formulas.

⚠️ In many write-ups the letter $k$ does two jobs: it counts training sequences, and it also names a state (as in
$b_k(j)$). We keep $K$/$k$ for sequences and $n$, $i$, $j$ for states.

In [ ]:
# ── 0.1 The two-dice HMM and two small helpers ──
STATES, FACES = DICE_HMM["states"], DICE_HMM["faces"]       # ('A', 'B'), (1, …, 6)
pi, trans, emit = DICE_HMM["pi"], DICE_HMM["A"], DICE_HMM["B"]   # π, a (rows = from), b (rows = state)
DIE_COLORS = ("tab:blue", "tab:orange")


def to_states(s: str) -> np.ndarray:
    """'AAB' → array([0, 0, 1])."""
    return np.array([STATES.index(ch) for ch in s])


def to_obs(s: str) -> np.ndarray:
    """'1,2,6' → array([0, 1, 5]) (faces → 0-based indices)."""
    return np.array([int(ch) - 1 for ch in s.split(",")])


def sample_hmm_many(K: int, T: int, pi, trans, emit, rng):
    """Sample K independent sequences of length T at once. Returns (Q, O), two K × T integer arrays."""
    n, m = emit.shape
    Q = np.empty((K, T), dtype=int)
    Q[:, 0] = rng.choice(n, size=K, p=pi)
    for t in range(1, T):                                     # inverse-CDF sampling, all K chains in parallel
        cum = np.cumsum(trans[Q[:, t - 1]], axis=1)
        Q[:, t] = np.minimum((rng.random(K)[:, None] >= cum).sum(axis=1), n - 1)
    cum = np.cumsum(emit[Q], axis=2)
    O = np.minimum((rng.random((K, T))[..., None] >= cum).sum(axis=2), m - 1)
    return Q, O


print("π =", pi, "\ntransitions:\n", trans, "\nemissions:\n", emit)

---
## 1 · Learning when the states are observed

> 📍 **Where we are.** [Module 05](05_likelihood_mle_map.ipynb) estimated a coin's bias by maximum likelihood;
> [Modules 08](08_knn_naive_bayes.ipynb) and [17](17_bayesian_networks_inference_learning.ipynb) estimated naïve-Bayes and
> Bayesian-network tables by counting. An HMM is a Bayesian network ([Module 19](19_hmm_inference.ipynb), Section 2), so when *every* variable is
> observed its MLE is again a set of count ratios. The graph structure is always known or fixed; only the parameters —
> the conditional probability tables (initial, transition and emission probabilities) — are learned.

### 1.1 The likelihood of labelled sequences

The data are $K$ independent labelled sequences $(O^1, Q^1), \dots, (O^K, Q^K)$. Independence means the likelihood is a
product over sequences, and each factor is the HMM joint probability of Module 19:

$$\log P\big((O^1,Q^1),\dots,(O^K,Q^K)\big) = \log\prod_{k=1}^{K}\Big[P(q_1^k)\,P(o_1^k\mid q_1^k)\prod_{t=2}^{T}P(q_t^k\mid q_{t-1}^k)\,P(o_t^k\mid q_t^k)\Big].$$

*In words:* the log-probability of all the data is the log of a product of start, transition and emission
probabilities — one start per sequence, one transition per neighbouring pair, one emission per symbol.

The log turns the product into a sum, and the sum falls apart into three groups that share no parameters:

$$= \underbrace{\sum_k \log \pi_{q_1^k}}_{\text{only initial}} \;+\; \underbrace{\sum_k\sum_{t=1}^{T}\log b_{q_t^k}(o_t^k)}_{\text{only emission}}
\;+\; \underbrace{\sum_k\sum_{t=2}^{T}\log a_{q_{t-1}^k q_t^k}}_{\text{only transition}}
\;=\; \sum_i N^{\text{start}}_i\log\pi_i + \sum_{i,\sigma} M_i(\sigma)\log b_i(\sigma) + \sum_{i,j} N_{ij}\log a_{ij}.$$

*In words:* the log-likelihood depends on the data only through **counts** — $N_i^{\text{start}}$ = how many sequences
start in $s_i$, $M_i(\sigma)$ = how often state $s_i$ emitted $\sigma$, $N_{ij}$ = how often $s_i$ was followed by $s_j$ —
and each group can be maximised on its own.

🧮 **Math refresher — the MLE of a categorical distribution is a count ratio.** With two categories (a coin, Module 05),
maximise $f(p) = N_A\log p + N_B\log(1-p)$: the derivative $N_A/p - N_B/(1-p)$ is zero at $\hat p = N_A/(N_A+N_B)$. With
$c$ categories, maximise $\sum_k N_k\log\theta_k$ subject to $\sum_k\theta_k = 1$. (Optional: add a Lagrange multiplier
$\lambda(1 - \sum_k\theta_k)$; setting the derivative $N_k/\theta_k - \lambda$ to zero gives $\theta_k = N_k/\lambda$, and
the constraint forces $\lambda = \sum_k N_k$.) Result: $\hat\theta_k = N_k / \sum_{k'} N_{k'}$. Each row of the transition
matrix and each row of the emission matrix is its own categorical distribution, so

$$\hat\pi_i = \frac{N^{\text{start}}_i}{K},\qquad \hat a_{ij} = \frac{N_{ij}}{\sum_{j'}N_{ij'}},\qquad \hat b_i(\sigma) = \frac{M_i(\sigma)}{\sum_{\sigma'}M_i(\sigma')}.$$

*In words:* the fraction of sequences that start in $s_i$; the fraction of departures from $s_i$ that go to $s_j$; the
fraction of $s_i$'s emissions that are $\sigma$.

### 1.2 A worked example: three labelled sequences

Consider our problem of rolling two dice A and B. We observe three sequences of rolls — `1,2,2,5,6,5,1`,
`1,3,2,5,6,5,2` and `3,2,1,3,6,5,4` — and, this time, the dice that produced them are revealed too (fully observed
data). How can we determine the initial, transition and emission probabilities?

In [ ]:
# ── 1.1 The three labelled example sequences ──
ex_states = ["AAABBAA", "AABBBBB", "BAABBAB"]
ex_rolls = ["1,2,2,5,6,5,1", "1,3,2,5,6,5,2", "3,2,1,3,6,5,4"]
Q_ex = np.array([to_states(s) for s in ex_states])       # K × T = 3 × 7
O_ex = np.array([to_obs(s) for s in ex_rolls])
for q, o in zip(ex_states, ex_rolls):
    print(" ".join(q), "   ", " ".join(o.split(",")))
print("K =", Q_ex.shape[0], "sequences of length T =", Q_ex.shape[1])

In [ ]:
# ── 1.2 Initial probabilities: look at the first state of every sequence ──
first = Q_ex[:, 0]
for i, s in enumerate(STATES):
    print(f"π_{s} = #sequences starting in {s} / K = {Fraction(int(np.sum(first == i)), len(first))} ≈ {np.mean(first == i):.3f}")

In [ ]:
# ── 1.3 Transitions: move a window of size 2 along each sequence ──
N_trans = np.zeros((2, 2), dtype=int)
for q in Q_ex:
    for prev, nxt in zip(q[:-1], q[1:]):                     # the window (q_{t-1}, q_t)
        N_trans[prev, nxt] += 1
print("transition counts N_ij (row = from, column = to):")
print(pd.DataFrame(N_trans, index=[f"from {s}" for s in STATES], columns=[f"to {s}" for s in STATES]))
for i, si in enumerate(STATES):
    for j, sj in enumerate(STATES):
        print(f"a_{si}{sj} = #{si}{sj} / (#{si}A + #{si}B) = {N_trans[i, j]}/{N_trans[i].sum()} = "
              f"{Fraction(int(N_trans[i, j]), int(N_trans[i].sum()))} ≈ {N_trans[i, j] / N_trans[i].sum():.3f}")
print("(fractions are shown in lowest terms, e.g. 3/9 = 1/3)")

In [ ]:
# ── 1.4 Emissions: count (state, face) pairs at the same position ──
M_emit = np.zeros((2, 6), dtype=int)
for q, o in zip(Q_ex, O_ex):
    for state, face in zip(q, o):
        M_emit[state, face] += 1
print("emission counts M_i(face):")
print(pd.DataFrame(M_emit, index=[f"die {s}" for s in STATES], columns=FACES))
print("\nMLE of the emission probabilities:")
print(pd.DataFrame([[str(Fraction(int(c), int(r.sum()))) for c in r] for r in M_emit],
                   index=[f"b_{s}" for s in STATES], columns=FACES))
print(f"\nb_A(5) = #A5 / #A = {M_emit[0, 4]}/{M_emit[0].sum()} = {M_emit[0, 4] / M_emit[0].sum():.2f}")

🔍 **What just happened?** Every number of the example, computed by counting:

* $\hat\pi_A = 2/3$, $\hat\pi_B = 1/3$ (two of the three sequences start with A);
* the 18 neighbouring pairs contain AA 5, AB 4, BA 3, BB 6 times, so $\hat a_{AA} = 5/9$, $\hat a_{AB} = 4/9$,
  $\hat a_{BA} = 1/3$, $\hat a_{BB} = 2/3$;
* die A was used 10 times and rolled 1, 1, 1, 1, 2, 2, 2, 3, 5, 5 — so $\hat b_A = (4, 3, 1, 0, 2, 0)/10$ and
  $\hat b_A(5) = \#A5/(\#A1+\dots+\#A6) = \#A5/\#A = 2/10$; die B was used 11 times, $\hat b_B = (0, 2, 2, 1, 3, 3)/11$.

⚠️ Transitions are counted with a moving window of size 2 over the *states* (#AA, #AB, #BA, #BB). Emissions pair each
state with the output **at the same position** — a window of size 1 over (state, output) pairs (#A1, …, #A6,
#B1, …, #B6). Mixing the two up is a common slip.

Let us wrap this into one reusable function and compare the estimates with the two-dice model of Module 19.

In [ ]:
# ── 1.5 A general counting-MLE function ──
def count_mle(Q, O, n: int, m: int, pseudo: float = 0.0):
    """MLE (pseudo = 0) or MAP-style smoothed estimate (pseudo > 0) of (π, a, b) from labelled K × T arrays Q, O."""
    N_start = np.bincount(Q[:, 0], minlength=n) + pseudo
    N_ij = np.zeros((n, n)) + pseudo
    np.add.at(N_ij, (Q[:, :-1].ravel(), Q[:, 1:].ravel()), 1)        # every (q_{t-1}, q_t) window
    M = np.zeros((n, m)) + pseudo
    np.add.at(M, (Q.ravel(), O.ravel()), 1)                           # every (q_t, o_t) pair
    return N_start / N_start.sum(), N_ij / N_ij.sum(1, keepdims=True), M / M.sum(1, keepdims=True)


pi_hat, trans_hat, emit_hat = count_mle(Q_ex, O_ex, 2, 6)
print("π̂ =", np.round(pi_hat, 3), "\nâ =\n", np.round(trans_hat, 3), "\nb̂ =\n", np.round(emit_hat, 3))

fig, axes = plt.subplots(1, 2, figsize=(10, 2.8), sharey=True)
for i, ax in enumerate(axes):
    ax.bar(np.array(FACES) - 0.2, emit[i], width=0.4, color="0.75", label="two-dice model (Module 19)")
    ax.bar(np.array(FACES) + 0.2, emit_hat[i], width=0.4, color=DIE_COLORS[i], label=f"MLE from {M_emit[i].sum()} rolls")
    ax.set_xticks(FACES); ax.set_xlabel("face"); ax.set_title(f"die {STATES[i]}: emission probabilities"); ax.legend()
axes[0].set_ylabel("probability")
plt.tight_layout(); plt.show()

🔍 **What just happened?** `count_mle` reproduces the hand counts. Compared with the two-dice model (grey bars) the
estimates are rough: 10 and 11 rolls are far too few to pin down six probabilities each, and some faces were simply never
seen — $\hat b_A(4) = \hat b_A(6) = \hat b_B(1) = 0$. (In fairness, the three example sequences were invented for the
counting exercise, not sampled from the two-dice model.) Is the count ratio really the *maximum*? Let us check numerically
for one parameter, $a_{AB}$: the transition part of the log-likelihood that involves row A is $5\log a_{AA} + 4\log a_{AB}$
with $a_{AA} = 1 - a_{AB}$.

In [ ]:
# ── 1.6 Check: the count ratio maximises the log-likelihood ──
x = np.linspace(0.01, 0.99, 981)
ll_row = N_trans[0, 0] * np.log(1 - x) + N_trans[0, 1] * np.log(x)
print(f"grid maximum at a_AB = {x[np.argmax(ll_row)]:.3f}   (count ratio 4/9 = {4 / 9:.3f})")


def complete_loglik(Q, O, pi_, trans_, emit_):
    """log P(O, Q) summed over labelled sequences (the quantity maximised by count_mle)."""
    with np.errstate(divide="ignore"):
        return (np.log(pi_[Q[:, 0]]).sum() + np.log(trans_[Q[:, :-1], Q[:, 1:]]).sum()
                + np.log(emit_[Q, O]).sum())


rng = np.random.default_rng(SEED + 1)
best = complete_loglik(Q_ex, O_ex, pi_hat, trans_hat, emit_hat)
others = [complete_loglik(Q_ex, O_ex, rng.dirichlet(np.ones(2)), rng.dirichlet(np.ones(2), 2),
                          rng.dirichlet(np.ones(6), 2)) for _ in range(2000)]
print(f"log-likelihood at the MLE: {best:.3f};  best of 2,000 random parameter sets: {max(others):.3f}")

fig, ax = plt.subplots(figsize=(5.5, 3))
ax.plot(x, ll_row, color="tab:blue"); ax.axvline(4 / 9, color="tab:red", ls="--", label="count ratio 4/9")
ax.set_xlabel(r"$a_{AB}$  (with $a_{AA} = 1 - a_{AB}$)"); ax.set_ylabel(r"$5\log a_{AA} + 4\log a_{AB}$")
ax.set_title("row-A transition log-likelihood"); ax.legend(); plt.tight_layout(); plt.show()

🔍 **What just happened?** The log-likelihood curve peaks exactly at $4/9 \approx 0.444$, and none of 2,000 random
parameter sets comes close to the MLE's log-likelihood (printed above). Counting is not a heuristic — it *is* maximum
likelihood.

⚠️ **Zero counts are dangerous.** Because $\hat b_A(6) = 0$, the fitted model says die A *never* rolls a 6. Any new
labelled sequence in which die A rolls a 6 gets probability exactly 0 ($\log = -\infty$), however well the rest fits.
The cure from Module 05: add **pseudo-counts** $\alpha$ to every count before normalising,
$\hat b_i(\sigma) = (M_i(\sigma) + \alpha)/(\sum_{\sigma'}M_i(\sigma') + m\alpha)$ — as if we had seen every face
$\alpha$ extra "imaginary" times. This is the MAP estimate under a Dirichlet prior with all parameters equal to
$\alpha + 1$ (the multi-category Beta of Module 05); $\alpha = 1$ is Laplace ("add-one") smoothing, and it gives
$\hat b_A(6) = (0+1)/(10+6) = 1/16$.

In [ ]:
# ── 1.7 Pseudo-counts rescue unseen events ──
Q_new, O_new = to_states("AABB")[None], to_obs("6,1,4,6")[None]      # die A rolls a 6 — never seen in training
for alpha in [0.0, 0.5, 1.0]:
    p_, a_, b_ = count_mle(Q_ex, O_ex, 2, 6, pseudo=alpha)
    print(f"pseudo-count {alpha}: b_A(6) = {b_[0, 5]:.4f},  log P(O_new, Q_new) = {complete_loglik(Q_new, O_new, p_, a_, b_):.3f}")

🔍 **What just happened?** Without pseudo-counts the new labelled sequence is "impossible" ($-\infty$); with $\alpha = 0.5$
or 1 it gets a finite, sensible log-probability. With 21 training rolls the choice of $\alpha$ matters a lot; with
thousands it hardly matters — the prior is swamped by the data, as in Module 05.

✏️ **Your turn.** (1) By hand, compute $\hat a_{BA}$ and $\hat b_B(5)$ with pseudo-count $\alpha = 1$ and check with
`count_mle(..., pseudo=1)`. (2) Add a fourth labelled sequence of your own to `ex_states`/`ex_rolls` and re-run 1.1–1.5.
Which estimates move the most, and why?

### 📘 Lecture notes — learning an HMM when the states are observed

**Setting.** The graph structure of an HMM is always known (fixed); learning means estimating the conditional
probability distributions — initial, transition and emission probabilities. If the hidden variables are observed:
maximum likelihood. Data: $K$ sequences $(O^1,Q^1),\dots,(O^K,Q^K)$ with $O^k=(o^k_1,\dots,o^k_T)$, $Q^k=(q^k_1,\dots,q^k_T)$.

**Likelihood.**

$$\log\prod_kP(O^k,Q^k)=\sum_k\log p(q_1^k)+\sum_k\sum_t\log p(o_t^k\mid q_t^k)+\sum_k\sum_{t\ge2}\log p(q_t^k\mid q_{t-1}^k).$$

*In words:* the first sum involves only initial probabilities, the second only emission probabilities, the third
only transition probabilities — differentiate with respect to each parameter (with a Lagrange multiplier for each
"sums to one" constraint), set to zero, and solve: closed-form count ratios.

**Worked example** (states | rolls): `AAABBAA` | 1,2,2,5,6,5,1 · `AABBBBB` | 1,3,2,5,6,5,2 · `BAABBAB` | 3,2,1,3,6,5,4.
- initial: $\pi_A=\#A/(\#A+\#B)$ over the first states of the $K$ sequences $=2/3$, $\pi_B=1/3$;
- transitions ($a_{ij}=p(q_t=s_j\mid q_{t-1}=s_i)$, moving window of size 2 → #AA = 5, #AB = 4, #BA = 3, #BB = 6):
  $a_{A,B}=\#AB/(\#AB+\#AA)=4/9$, $a_{AA}=5/9$, $a_{BA}=1/3$, $a_{BB}=2/3$;
- emissions ($b_i(o_t)=P(o_t\mid q_t=s_i)$, counts #A1…#A6, #B1…#B6):
  $b_A(5)=\#A5/(\#A1+\dots+\#A6)=\#A5/\#A=2/10$; $\hat b_A=(4,3,1,0,2,0)/10$, $\hat b_B=(0,2,2,1,3,3)/11$.

**Zero counts.** An unseen event gets probability 0; pseudo-counts $\alpha$ (a Dirichlet prior, MAP) fix it:
$\hat b_i(\sigma)=(M_i(\sigma)+\alpha)/(\sum_{\sigma'}M_i(\sigma')+m\alpha)$.

> 🎓 **Exam tip.** Count transitions per *row*: $a_{AB}$ divides by the number of departures **from A** ($\#AA+\#AB$ = 9), not by all 18 pairs and not by the number of A's (10 — the last A of a sequence has no successor).

---
## 2 · When the states are hidden: a chicken-and-egg problem

> 📍 **Where we are.** Section 1 needed the labels. Usually we have only the rolls (or the DNA). [Module 13](13_mixture_models_em.ipynb) met the same
> situation for mixture models: no cluster labels, so no counts.

In most cases we do not know which states generated the outputs. The dilemma:

* *if we knew the states*, learning would be easy — count (Section 1);
* *if we knew the parameters*, inferring the states would be easy — forward–backward or Viterbi (Module 19);
* we know **neither**.

A tempting shortcut is to *guess* the labels with a simple rule and then count. Let us try it on simulated rolls for
which we secretly know the truth.

In [ ]:
# ── 2.1 Guess the labels with a rule, then count — what goes wrong? ──
rng = np.random.default_rng(SEED + 2)
Q_train, O_train = sample_hmm_many(40, 100, pi, trans, emit, rng)        # SIMULATED: 40 sequences × 100 rolls
print(f"SIMULATED training data: {O_train.size:,} rolls; die B used {np.mean(Q_train == 1):.1%} of the time")

Q_rule = (O_train >= 3).astype(int)                 # rule: faces 4–6 (indices 3–5) → die B, faces 1–3 → die A
print(f"accuracy of the rule's labels: {accuracy(Q_train.ravel(), Q_rule.ravel()):.3f}")
for name, Q_lab in [("rule labels", Q_rule), ("true labels (oracle)", Q_train)]:
    p_, a_, b_ = count_mle(Q_lab, O_train, 2, 6)
    print(f"\ncounting with {name}:\n  a_AA = {a_[0, 0]:.2f}, a_BB = {a_[1, 1]:.2f}\n  b_A = {np.round(b_[0], 2)}\n  b_B = {np.round(b_[1], 2)}")

🔍 **What just happened?** The rule labels about 70 % of the rolls correctly (0.699), but counting with
them gives absurd parameters: die A "never" rolls 4–6 and die B "never" rolls 1–3, and the stay-probabilities come out as 0.54
instead of 0.8 because the rule's labels flip whenever the face crosses the threshold. With the true
labels (which we will not have in practice) counting recovers emissions and transitions close to the true model. A
**hard** guess throws away the uncertainty about each label — a 3 is *probably* die A, not *certainly*.

### 2.1 The EM idea: count with probabilities

The **expectation–maximisation** algorithm (Dempster, Laird & Rubin 1977; for HMMs, Baum *et al.* 1970 — hence
**Baum–Welch**) is appropriate for problems with "missing values" — here, the hidden states. It fills in the missing
variables by inference and replaces actual counts by **expected counts** under the current parameters. Formally, each
iteration maximises the expected complete-data log-likelihood

$$\theta^{\text{new}} = \arg\max_\theta\; \mathbb E_{Q\sim P(Q\mid O,\theta^{\text{old}})}\big[\log P(O, Q \mid \theta)\big].$$

*In words:* average the log-likelihood of the *labelled* data over all labellings, weighting each labelling by how
probable it is under the current parameters, and maximise that average.

Section 1 showed that $\log P(O,Q\mid\theta) = \sum(\text{count})\times\log(\text{parameter})$. An expectation of a sum is
the sum of expectations, so the expected log-likelihood is $\sum \mathbb E[\text{count}]\times\log(\text{parameter})$ —
the same form, with expected counts. Its maximiser is therefore **the same count ratios, with expected counts**:
*E step* — fill in the missing variables with their expected values (expected counts, computed with forward–backward);
*M step* — regular maximum-likelihood estimation (count ratios) using those values. Repeat.

**What EM guarantees.** Each iteration never decreases the observed-data likelihood $P(O\mid\theta)$. It converges to a
stationary point — usually a local maximum — with **no guarantee of finding the global maximum**; different starting
points can end in different places (Section 4). ⚠️ You may read that EM is "guaranteed to converge, though only to a
local minimum"; since EM *climbs* the likelihood, the right word is local **maximum** (or, rarely, a saddle point).

**HMMs and mixture models.** If every row of the transition matrix equals the initial distribution $\boldsymbol\pi$
($a_{ij} = \pi_j$ for all $i$), the next state ignores the current one: the states are drawn independently and the HMM *is* a mixture
model (Module 13). If the M-step also ties the rows (one shared row $=\boldsymbol\pi$, re-estimated from the soft labels of
all positions), Baum–Welch reduces to EM for a mixture: $S_t(i)$ are the responsibilities. An HMM is "a mixture
model whose labels are chained over time".

---
## 3 · The E-step and the M-step

> 📍 **Where we are.** We need expected versions of the counts of Section 1. Module 19 already computed the key
> ingredient — posterior state probabilities from the forward and backward variables.

### 3.1 Expected counts

With labels, we counted states (#A, #B) and pairs of consecutive states (#AA, #AB, …). Without labels, we need the
probability of each state and each pair at each position, given all observations:

$$S_t(i) = P(q_t = s_i \mid o_1,\dots,o_T), \qquad S_t(i,j) = P(q_t = s_i,\; q_{t+1} = s_j \mid o_1,\dots,o_T).$$

*In words:* $S_t(i)$ is the "soft label" of position $t$; $S_t(i,j)$ the probability that the transition $s_i \to s_j$
happened between $t$ and $t+1$. An expected count is a sum of such probabilities — e.g. the expected number of A→B
transitions is $\sum_t S_t(A,B)$, just as the actual number is $\sum_t \mathbb 1[q_t = A, q_{t+1} = B]$.

From Module 19 (Section 8): $S_t(i) = \alpha_t(i)\beta_t(i) / \sum_j\alpha_t(j)\beta_t(j)$. For the pair probability:

$$
\begin{aligned}
P(q_t=s_i, q_{t+1}=s_j, o_1..o_T) &= P(o_{t+2}..o_T \mid o_1..o_{t+1}, q_t=s_i, q_{t+1}=s_j)\, P(o_1..o_{t+1}, q_t=s_i, q_{t+1}=s_j) &&\text{[product rule]}\\
&= \beta_{t+1}(j)\; P(o_{t+1}\mid o_1..o_t, q_t=s_i, q_{t+1}=s_j)\; P(o_1..o_t, q_t=s_i, q_{t+1}=s_j) &&\text{[Markov; product rule]}\\
&= \beta_{t+1}(j)\; b_j(o_{t+1})\; P(q_{t+1}=s_j \mid o_1..o_t, q_t=s_i)\; P(o_1..o_t, q_t=s_i) &&\text{[Markov; product rule]}\\
&= \beta_{t+1}(j)\; b_j(o_{t+1})\; a_{ij}\; \alpha_t(i) &&\text{[Markov; definition of } \alpha_t].
\end{aligned}
$$

*In words:* peel off the future observations ($\beta_{t+1}$), then the next emission and the transition; what remains
is the past ending in $s_i$ ($\alpha_t$).

Dividing by $P(O) = \sum_j \alpha_t(j)\beta_t(j)$:

$$S_t(i,j) = \frac{\alpha_t(i)\; a_{ij}\; b_j(o_{t+1})\; \beta_{t+1}(j)}{\sum_{j'}\alpha_t(j')\,\beta_t(j')}.$$

*In words:* the probability of using the arrow $s_i\to s_j$ between $t$ and $t+1$ is "all the past, ending in $s_i$" ×
"take the arrow" × "emit the next symbol from $s_j$" × "all the future, starting from $s_j$", normalised. (A way to
remember the derivation: first peel off $\beta_{t+1}$, then the emission and the transition, and you are left with
$\alpha_t$.)

### 3.2 The same with scaling

For long sequences we use Module 19's scaled variables: $\hat\alpha_t(i) = P(q_t=s_i\mid o_1..o_t)$ and normalisers
$c_t = P(o_t\mid o_1..o_{t-1})$, with $\hat\beta_t$ scaled by the same $c$'s. Then

$$S_t(i) = \hat\alpha_t(i)\,\hat\beta_t(i), \qquad S_t(i,j) = \frac{\hat\alpha_t(i)\, a_{ij}\, b_j(o_{t+1})\,\hat\beta_{t+1}(j)}{c_{t+1}}, \qquad \log P(O) = \sum_t \log c_t .$$

*In words:* the scaled tables give the soft labels and soft transitions directly, with no division by a tiny $P(O)$.

The function below does this for $K$ sequences of equal length **at once** (the arrays carry an extra leading axis $k$);
a single sequence is simply $K = 1$.

In [ ]:
# ── 3.1 Scaled forward–backward for K sequences at once: the E-step ──
def forward_backward(O, pi, trans, emit):
    """E-step for a K × T array of sequences.

    Returns S (K, T, n) with S[k, t, i] = P(q_t = s_i | O^k), S2 (K, T−1, n, n) with
    S2[k, t, i, j] = P(q_t = s_i, q_{t+1} = s_j | O^k), and the total log-likelihood Σ_k log P(O^k).
    """
    K, T = O.shape
    n = len(pi)
    E = emit[:, O].transpose(1, 2, 0)                       # E[k, t, i] = b_i(o_t^k)
    alpha, beta, c = np.zeros((K, T, n)), np.ones((K, T, n)), np.zeros((K, T))
    a = pi * E[:, 0]                                        # forward: α_1(i) = π_i b_i(o_1), then normalise
    c[:, 0] = a.sum(1); alpha[:, 0] = a / c[:, :1]
    for t in range(1, T):
        a = (alpha[:, t - 1] @ trans) * E[:, t]             # Σ_j α̂_{t-1}(j) a_ji · b_i(o_t)
        c[:, t] = a.sum(1); alpha[:, t] = a / c[:, t:t + 1]
    for t in range(T - 2, -1, -1):                          # backward, scaled by the same c's
        beta[:, t] = ((E[:, t + 1] * beta[:, t + 1]) @ trans.T) / c[:, t + 1:t + 2]
    S = alpha * beta                                        # soft labels; each S[k, t] sums to 1
    S2 = (alpha[:, :-1, :, None] * trans[None, None]        # α̂_t(i) · a_ij
          * (E[:, 1:] * beta[:, 1:])[:, :, None, :]          # · b_j(o_{t+1}) β̂_{t+1}(j)
          / c[:, 1:, None, None])                            # / c_{t+1}
    return S, S2, np.log(c).sum()

In [ ]:
# ── 3.2 Check against brute-force enumeration of all 2^6 paths ──
o6 = to_obs("1,6,6,2,5,1")
S, S2, ll = forward_backward(o6[None], pi, trans, emit)
paths = np.array(list(itertools.product(range(2), repeat=6)))
joint = pi[paths[:, 0]] * emit[paths[:, 0], o6[0]]
for t in range(1, 6):
    joint = joint * trans[paths[:, t - 1], paths[:, t]] * emit[paths[:, t], o6[t]]
post = joint / joint.sum()
S_bf = np.array([[post[paths[:, t] == i].sum() for i in range(2)] for t in range(6)])
S2_bf = np.array([[[post[(paths[:, t] == i) & (paths[:, t + 1] == j)].sum() for j in range(2)] for i in range(2)]
                  for t in range(5)])
print(f"log P(O): forward–backward {ll:.10f} | brute force {np.log(joint.sum()):.10f}")
print(f"max |S_t(i) − brute force|   = {np.abs(S[0] - S_bf).max():.1e}")
print(f"max |S_t(i,j) − brute force| = {np.abs(S2[0] - S2_bf).max():.1e}")
print(f"consistency: Σ_j S_t(i,j) = S_t(i)?  max difference {np.abs(S2[0].sum(2) - S[0, :-1]).max():.1e}")
print("soft labels P(q_t = B | O) for rolls 1 6 6 2 5 1:", np.round(S[0, :, 1], 3))

🔍 **What just happened?** The scaled forward–backward pass reproduces the exact posteriors of all 64 paths to rounding
error, both for single positions and for pairs, and the pair probabilities sum to the single-position ones (summing out
$q_{t+1}$ is the law of total probability). The soft labels say what a careful human would: the two 6s are *probably* die
B (0.63 and 0.69), the 1s at both ends *probably* die A (0.24 and 0.33 for B), and the 2 and 5 in between are a
toss-up (≈ 0.5) — nothing is certain.

Now apply the E-step to the three example sequences, pretending we cannot see the dice, using the two-dice model's
parameters. How do the *expected* counts compare with the actual counts of Section 1?

In [ ]:
# ── 3.3 Expected vs actual counts on the three example sequences ──
S, S2, _ = forward_backward(O_ex, pi, trans, emit)
exp_trans = S2.sum(axis=(0, 1))                              # Σ_k Σ_t S_t(i, j)
exp_state = S.sum(axis=(0, 1))                               # Σ_k Σ_t S_t(i)
rows = [("#AA", N_trans[0, 0], exp_trans[0, 0]), ("#AB", N_trans[0, 1], exp_trans[0, 1]),
        ("#BA", N_trans[1, 0], exp_trans[1, 0]), ("#BB", N_trans[1, 1], exp_trans[1, 1]),
        ("#A (all positions)", (Q_ex == 0).sum(), exp_state[0]), ("#B (all positions)", (Q_ex == 1).sum(), exp_state[1]),
        ("#A5", M_emit[0, 4], S[..., 0][O_ex == 4].sum()), ("#B6", M_emit[1, 5], S[..., 1][O_ex == 5].sum())]
print(pd.DataFrame(rows, columns=["count", "actual (states shown)", "expected (states hidden)"]).round(2).to_string(index=False))

🔍 **What just happened?** Expected counts are fractional (a transition can be "0.6 of an A→B"), and they add up
correctly: 18 expected transitions, 21 expected states. They differ from the example's labels because the model and
the labels disagree: with a stay-probability of 0.8 the model expects far fewer switches of die (expected
#AB + #BA ≈ 2.69 + 1.29 ≈ 4.0) than the 7 switches in the labels. That is fine: the expected counts are what the
*current model* believes, and EM will update the model in the direction of the data.

### 3.3 The M-step

Plug the expected counts into the count-ratio formulas of Section 1. Summing also over the $K$ sequences:

$$\hat a_{ij} = \frac{\hat n(i,j)}{\sum_{j'}\hat n(i,j')},\quad \hat n(i,j) = \sum_k\sum_{t=1}^{T-1} S^k_t(i,j);\qquad
\hat b_i(\sigma) = \frac{B_i(\sigma)}{\sum_{\sigma'}B_i(\sigma')},\quad B_i(\sigma) = \sum_k\sum_{t:\,o^k_t=\sigma} S^k_t(i);\qquad
\hat\pi_i = \frac1K\sum_k S^k_1(i).$$

*In words:* the new transition probability is the expected number of $i\to j$ transitions over the expected number of
departures from $i$ — with labels this was $a_{A,B}=\#AB/(\#AB+\#AA)$, and $\#AB=\sum_t\mathbb 1[(q_t,q_{t+1})=AB]$ is
replaced by $\sum_t S_t(A,B)$; the new emission probability (a multinomial) is the expected number of times $s_i$ emitted
$\sigma$ over the expected number of times $s_i$ was visited; the new initial probability is the average soft label of
the first position across sequences — initial probabilities are deduced from the first observation of each of the
multiple sequences.

In [ ]:
# ── 3.4 The M-step, and a check that it reduces to counting when the labels are certain ──
def m_step(O, S, S2, m: int):
    """Count-ratio estimates from expected counts: returns new (π, a, b)."""
    pi_new = S[:, 0].mean(axis=0)                                    # average soft label at t = 1
    n_ij = S2.sum(axis=(0, 1))                                       # expected transition counts
    B = np.einsum("kti,ktm->im", S, np.eye(m)[O])                    # B[i, σ] = Σ_{k,t: o_t = σ} S_t(i)
    return pi_new, n_ij / n_ij.sum(axis=1, keepdims=True), B / B.sum(axis=1, keepdims=True)


S_hard = np.eye(2)[Q_ex]                                          # one-hot "soft" labels = the true labels
S2_hard = S_hard[:, :-1, :, None] * S_hard[:, 1:, None, :]           # one-hot pairs
p_h, a_h, b_h = m_step(O_ex, S_hard, S2_hard, 6)
print("M-step with certain labels equals counting:",
      np.allclose(p_h, pi_hat), np.allclose(a_h, trans_hat), np.allclose(b_h, emit_hat))

🔍 **What just happened?** When the "soft" labels are 0/1 (certainty), expected counts *are* actual counts and the M-step
returns exactly the estimates of Section 1. Baum–Welch is counting with uncertainty: the E-step spreads each position's
vote over the states in proportion to their posterior probability.

✏️ **Your turn.** Run one full EM step by hand on the example sequences: `S, S2, ll = forward_backward(O_ex, pi, trans, emit)`,
then `m_step(O_ex, S, S2, 6)`. Compare the new $a_{AB}$ with the model's 0.2 and with the labelled estimate 4/9.
Run the E-step again with the new parameters: did the log-likelihood `ll` go up?

---
## 4 · Baum–Welch from scratch

> 📍 **Where we are.** We have an E-step (`forward_backward`) and an M-step (`m_step`). Baum–Welch alternates them
> until the log-likelihood stops improving.

**The complete EM algorithm for learning the parameters of an HMM (Baum–Welch).** *Inputs:* the observation
sequences and the number of states $n$.
1. Guess initial parameters $\pi, a, b$.
2. **E-step:** compute $S_t(i)$ and $S_t(i,j)$ for every sequence (forward–backward).
3. **M-step:** re-estimate $\pi, a, b$ from the expected counts.
4. If the log-likelihood improved by more than a small tolerance, go back to 2; otherwise output the model.

In [ ]:
# ── 4.1 Baum–Welch ──
def random_init(n: int, m: int, rng, stay=None):
    """Random starting parameters: uniform π; Dirichlet(1) rows, or 'sticky' transitions with P(stay) = stay."""
    pi0 = np.full(n, 1.0 / n)
    if n == 1:
        trans0 = np.ones((1, 1))
    elif stay is None:
        trans0 = rng.dirichlet(np.ones(n), size=n)
    else:
        trans0 = np.full((n, n), (1 - stay) / (n - 1)); np.fill_diagonal(trans0, stay)
    return pi0, trans0, rng.dirichlet(np.ones(m), size=n)


def baum_welch(O, pi0, trans0, emit0, max_iter: int = 300, tol: float = 1e-3):
    """EM for an HMM on a K × T array of sequences. Returns (π, a, b, log-likelihood history)."""
    pi_, trans_, emit_ = pi0.copy(), trans0.copy(), emit0.copy()
    m = emit_.shape[1]
    history = []
    for it in range(max_iter):
        S, S2, ll = forward_backward(O, pi_, trans_, emit_)      # E-step (+ log-likelihood of current parameters)
        history.append(ll)
        if it > 0 and history[-1] - history[-2] < tol:          # converged: the returned parameters match history[-1]
            break
        pi_, trans_, emit_ = m_step(O, S, S2, m)                # M-step
    return pi_, trans_, emit_, np.array(history)

In [ ]:
# ── 4.2 Learn the dice from the 4,000 SIMULATED rolls of Section 2 (labels hidden) ──
rng = np.random.default_rng(SEED + 4)
init = random_init(2, 6, rng, stay=0.9)
pi_bw, trans_bw, emit_bw, hist = baum_welch(O_train, *init, max_iter=500, tol=1e-4)
ll_true = forward_backward(O_train, pi, trans, emit)[2]
print(f"iterations: {len(hist)};  log-likelihood {hist[0]:.1f} → {hist[-1]:.1f}")
print(f"smallest change between consecutive iterations: {min(np.diff(hist)):+.2e}  (never negative: EM never goes down)")
print(f"log-likelihood under the TRUE parameters: {ll_true:.1f}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.arange(1, len(hist) + 1), hist, color="tab:blue", label="Baum–Welch")
ax.axhline(ll_true, color="tab:red", ls="--", label="true parameters")
ax.set_xscale("log"); ax.set_xlabel("iteration (log scale)"); ax.set_ylabel("log P(O | θ)")
ax.set_title("SIMULATED rolls: the log-likelihood climbs monotonically"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** Starting from random emissions, the log-likelihood rises from −7340 to −7092 in 233
iterations, and every single change is positive (the smallest is printed) — the EM guarantee in action. Most of the
gain comes in the first 10 iterations; the rest is a slow creep, typical of EM. It ends *above* the
log-likelihood of the true parameters: the MLE fits *this particular sample* better than the truth does, a mild form of
overfitting that shrinks as data grow. Now compare the learned parameters with the truth. One subtlety first.

⚠️ **Label switching.** The likelihood does not care what we *call* the states: swapping the names "A" and "B" everywhere
(rows and columns of `trans`, rows of `emit`, entries of `pi`) gives exactly the same $P(O)$. EM may therefore return die
B as "state 0". Before comparing with the truth we match learned states to true ones (here: the order that best matches
the emission tables).

In [ ]:
# ── 4.3 Match the labels, then compare with the truth ──
def match_states(pi_, trans_, emit_, emit_true):
    """Permute the learned states so that their emission tables best match the true ones (all permutations)."""
    n = len(pi_)
    best = min(itertools.permutations(range(n)), key=lambda p: np.abs(emit_[list(p)] - emit_true).sum())
    p = list(best)
    return pi_[p], trans_[np.ix_(p, p)], emit_[p], p


pi_m, trans_m, emit_m, perm = match_states(pi_bw, trans_bw, emit_bw, emit)
print("state order after matching:", perm)
print("transitions — learned:\n", np.round(trans_m, 3), "\n   true:\n", trans)
print("emissions — learned:\n", np.round(emit_m, 3), "\n   true:\n", emit)
print(f"max |error|: transitions {np.abs(trans_m - trans).max():.3f}, emissions {np.abs(emit_m - emit).max():.3f}")
q_dec = forward_backward(O_train, pi_m, trans_m, emit_m)[0].argmax(2)          # decode with the matched model
q_true_model = forward_backward(O_train, pi, trans, emit)[0].argmax(2)
print(f"posterior decoding accuracy — LEARNED model: {accuracy(Q_train.ravel(), q_dec.ravel()):.3f}, "
      f"TRUE model: {accuracy(Q_train.ravel(), q_true_model.ravel()):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 2.8), sharey=True)
for i, ax in enumerate(axes):
    ax.bar(np.array(FACES) - 0.2, emit[i], 0.4, color="0.75", label="truth")
    ax.bar(np.array(FACES) + 0.2, emit_m[i], 0.4, color=DIE_COLORS[i], label="Baum–Welch (no labels)")
    ax.set_xticks(FACES); ax.set_xlabel("face"); ax.set_title(f"die {STATES[i]}"); ax.legend()
axes[0].set_ylabel("emission probability")
plt.tight_layout(); plt.show()

🔍 **What just happened?** From 4,000 unlabelled rolls, Baum–Welch recovers the shape of both dice — A favours 1, B
favours 6 — with emission errors of at most 0.03. The stay-probabilities come out too high (0.87–0.88 instead of 0.8),
although counting with the true labels gave 0.79–0.80 on the very same rolls (Section 2): when the dice are similar,
the likelihood is nearly flat in the "stickiness" direction, so the unlabelled data pin the transitions down far less
precisely than labelled data would. Decoding the hidden dice with the learned model is nearly as accurate as decoding with the true
model (0.716 vs 0.735) — and neither is close to perfect, because the dice are hard to tell apart roll by roll. Next, two practical issues: where EM ends up depends on where it starts, and how
much data it needs.

In [ ]:
# ── 4.4 Random restarts: local optima and plateaus ──
fig, ax = plt.subplots(figsize=(6.5, 3.2))
finals = []
for r in range(8):
    init_r = random_init(2, 6, np.random.default_rng(100 + r))          # Dirichlet(1) transitions: not sticky
    p_r, a_r, b_r, h_r = baum_welch(O_train, *init_r, max_iter=400, tol=1e-3)
    finals.append((r, h_r[-1], len(h_r), a_r[0, 0], a_r[1, 1]))
    ax.plot(np.arange(1, len(h_r) + 1), h_r, lw=1.2, label=f"start {r}")
ax.axhline(ll_true, color="k", ls="--", lw=1, label="true parameters")
ax.set_xscale("log"); ax.set_ylim(ll_true - 60, hist[-1] + 8)
ax.set_xlabel("iteration (log scale)"); ax.set_ylabel("log P(O | θ)"); ax.legend(ncol=3, fontsize=7)
ax.set_title("8 random starts on the same SIMULATED data"); plt.tight_layout(); plt.show()
print(pd.DataFrame(finals, columns=["start", "final log-lik", "iterations", "a_00", "a_11"]).round(2).to_string(index=False))

🔍 **What just happened?** The eight runs do not all agree. Four starts (1, 3, 4, 6) reach the same best log-likelihood,
−7092.3, with "sticky" dice (both stay-probabilities ≈ 0.87, like Section 4.2). The other four stop 23–26 **nats** lower
(a *nat* is one unit of natural-log likelihood: 23 nats lower means the data are $e^{23} \approx 10^{10}$ times less
probable under that model) with
quite different explanations — start 7, for example, ends with stay-probabilities 0.16 and 0.15, i.e. dice that switch
at almost every roll — either at a genuine local optimum or still crawling along a nearly flat stretch of the
likelihood surface when the stopping rule (or the 400-iteration limit) fires. Curves that stay flat for dozens of iterations and then jump are the
signature of such **plateaus**. Practical rules: run EM from several starting points and keep the best log-likelihood;
use a tight tolerance (or a maximum number of iterations large enough to escape plateaus); and initialise sensibly
when you can (in Section 4.2 we started from "sticky" transitions, as biology often suggests — states persist).

⚠️ EM's guarantee is about *not going down*, not about reaching the top.

### 4.1 How much data does Baum–Welch need?

In [ ]:
# ── 4.5 Learning curve: parameter error vs number of rolls ──
sizes, err_emit, err_trans = [400, 1600, 6400, 25600], [], []
for N in sizes:
    Qd, Od = sample_hmm_many(N // 100, 100, pi, trans, emit, np.random.default_rng(N))
    fits = [baum_welch(Od, *random_init(2, 6, np.random.default_rng(N + r), stay=0.9)) for r in range(2)]
    p_b, a_b, b_b, h_b = max(fits, key=lambda f: f[3][-1])            # keep the best of 2 starts
    _, a_b, b_b, _ = match_states(p_b, a_b, b_b, emit)
    err_emit.append(np.abs(b_b - emit).max()); err_trans.append(np.abs(a_b - trans).max())
    print(f"N = {N:6,d} rolls: max emission error {err_emit[-1]:.3f}, max transition error {err_trans[-1]:.3f}")

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.loglog(sizes, err_emit, "o-", color="tab:blue", label="emissions")
ax.loglog(sizes, err_trans, "s-", color="tab:orange", label="transitions")
ax.loglog(sizes, err_emit[0] * np.sqrt(sizes[0] / np.array(sizes)), "k--", lw=1, label=r"$\propto 1/\sqrt{N}$")
ax.set_xlabel("number of SIMULATED rolls N"); ax.set_ylabel("max |learned − true|")
ax.set_title("Baum–Welch needs a lot of data"); ax.legend(); plt.tight_layout(); plt.show()

🔍 **What just happened?** With 400 rolls the learned probabilities can be off by about 0.2 — useless for telling the
dice apart. The error falls roughly like $1/\sqrt N$ (dashed line; the points are noisy because each size is a single
simulated data set), so every halving of the error costs four times more data. Hidden-state models are data-hungry:
the labels must be *inferred* before they can be counted, and when the states are similar, the information about them
per observation is small. Real applications (gene finders, profile HMMs) learn from thousands to millions of symbols,
often with labelled examples or informative priors to help.

✏️ **Your turn.** Make the dice easier to tell apart — e.g. die B with `[.02, .02, .02, .02, .02, .9]` — simulate new
training data and re-run 4.2–4.3. How many iterations does EM need now, and how large are the errors?

### 📘 Lecture notes — hidden states: EM, the Baum–Welch algorithm

**The dilemma.** Usually the states are unobserved. Had we known them, MLE would be easy (count); had we known the
parameters, inference would give the states (forward–backward, Viterbi). Unknown: both parameters and hidden
variables — a latent-variable model whose unobserved columns we impute with inference.

**EM** (for problems with missing values): assume the complete-data log-likelihood and maximise its *expectation*
under $p(Q\mid O,\text{parameters})$:
$\theta^{\text{new}}=\arg\max_\theta\mathbb E_{Q\sim p(Q\mid O,\theta^{\text{old}})}[\log P(O,Q\mid\theta)]$.
E step: fill in the missing variables with expected values; M step: regular MLE with those values. EM converges,
but only to a local optimum (a local **maximum** of the likelihood).

**E step = expected counts.** With labels we needed actual counts (#A, #B for initial and emission probabilities;
#AA, #AB, #BA, #BB for transitions). Without labels we compute, for all $t,i,j$,

$$S_t(i)=P(q_t=s_i\mid o_1..o_T)=\frac{\alpha_t(i)\beta_t(i)}{\sum_j\alpha_t(j)\beta_t(j)},\qquad
S_t(i,j)=P(q_t=s_i,q_{t+1}=s_j\mid o_1..o_T)=\frac{\alpha_t(i)\,a_{ij}\,b_j(o_{t+1})\,\beta_{t+1}(j)}{\sum_j\alpha_t(j)\beta_t(j)}.$$

*In words:* soft labels and soft transitions, from one forward and one backward pass — *inference as a subroutine*.

**M step = counting with expected counts.** $a_{i,j}=\hat n(i,j)/\sum_k\hat n(i,k)$ with $\hat n(i,j)=\sum_tS_t(i,j)$
(vs $\sum_t\mathbb 1[(q_t,q_{t+1})=(i,j)]$ when labels are known); emissions (multinomial):
$B_i(\sigma)=\sum_{t:o_t=\sigma}S_t(i)$, $b_i(\sigma)=B_i(\sigma)/\sum_{\sigma'}B_i(\sigma')$; initial probabilities from the
first observation of each of the multiple sequences, $\hat\pi_i=\frac1K\sum_kS^k_1(i)$.

**Baum–Welch.** Inputs: the observation sequences and the number of states. (1) Guess initial transition and
emission parameters; (2) E step: $S_t(i)$, $S_t(i,j)$; (3) M step; (4) converged? If not, go to (2); (5) output the
model. In practice: the log-likelihood never decreases; restart from several initialisations; states are identified
only up to relabelling.

> 🎓 **Exam tip.** Know the two E-step quantities and their formulas in terms of $\alpha$, $\beta$, $a$, $b$, and be able to write the M-step as "expected count / expected total". A favourite question: what does Baum–Welch become when the transition rows are all equal and tied in the M-step? (EM for a mixture model.)

---
## 5 · How many hidden states?

> 📍 **Where we are.** Baum–Welch needs $n$ as an input. How do we choose it? More states means a more complex model —
> and overfitting. This is a model-selection problem, previewing [Module 23](23_model_selection.ipynb).

More states always fit the *training* data at least as well: an $(n+1)$-state HMM can imitate any $n$-state HMM. So the
training log-likelihood cannot choose $n$ — it would always pick the largest. Two standard remedies:

* **held-out log-likelihood** — fit on training sequences, evaluate $\log P(O_{\text{test}}\mid\hat\theta)$ on new
  sequences (cross-validation, Module 23);
* **BIC** (Schwarz 1978) — penalise the number of free parameters $d$:

$$\mathrm{BIC} = -2\log P(O\mid\hat\theta) + d\,\log N,\qquad d = (n-1) + n(n-1) + n(m-1),$$

*In words:* lower is better; each extra free parameter must improve twice the log-likelihood by at least $\log N$
($N$ = number of observations) to be worth it. A third option are nonparametric Bayesian models (e.g. the
"infinite HMM"), which let the data decide how many states are used.

In [ ]:
# ── 5.1 Fit n = 1…5 states; compare training log-likelihood, held-out log-likelihood and BIC ──
_, O_test = sample_hmm_many(40, 100, pi, trans, emit, np.random.default_rng(SEED + 50))   # 4,000 new SIMULATED rolls


def fit_best(O, n, seeds):
    """Baum–Welch from several sticky random starts; keep the fit with the highest training log-likelihood."""
    fits = [baum_welch(O, *random_init(n, 6, np.random.default_rng(s), stay=0.9 if n > 1 else None)) for s in seeds]
    return max(fits, key=lambda f: f[3][-1])


rows = []
for n_states in range(1, 6):
    p_n, a_n, b_n, h_n = fit_best(O_train, n_states, seeds=[500 + 10 * n_states + r for r in range(3)])
    d = (n_states - 1) + n_states * (n_states - 1) + n_states * 5
    rows.append((n_states, d, h_n[-1], forward_backward(O_test, p_n, a_n, b_n)[2], -2 * h_n[-1] + d * np.log(O_train.size)))
sel = pd.DataFrame(rows, columns=["n states", "free params d", "train log-lik", "held-out log-lik", "BIC"])
print(sel.round(1).to_string(index=False))
print(f"\nchosen n — held-out log-lik: {sel.loc[sel['held-out log-lik'].idxmax(), 'n states']},  "
      f"BIC: {sel.loc[sel['BIC'].idxmin(), 'n states']}")

In [ ]:
# ── 5.2 Picture the trade-off ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(sel["n states"], sel["train log-lik"] / O_train.size, "o-", label="training")
ax1.plot(sel["n states"], sel["held-out log-lik"] / O_test.size, "s-", label="held-out")
ax1.set_xlabel("number of hidden states n"); ax1.set_ylabel("log-likelihood per roll"); ax1.legend()
ax1.set_title("more states always fit the training data better")
ax2.plot(sel["n states"], sel["BIC"], "o-", color="tab:green")
ax2.set_xlabel("number of hidden states n"); ax2.set_ylabel("BIC (lower = better)")
ax2.set_title(f"BIC with N = {O_train.size:,} rolls")
for ax in (ax1, ax2):
    ax.set_xticks(range(1, 6))
plt.tight_layout(); plt.show()

🔍 **What just happened? (An honest result.)** The training log-likelihood increases with every added state, as it
must. The held-out log-likelihood is highest at the true number of states, $n = 2$; every larger model predicts new
data worse — extra states fit noise. But BIC, with 4,000 rolls, prefers $n = 1$ — a single die! The two dice are so
similar that the 2-state model gains only about 27 nats of log-likelihood over one die (−7092.3 vs −7118.9 in the
table), while its 8 extra parameters cost $8\times\log(4000)/2 \approx 33$ nats under BIC's penalty. BIC is deliberately conservative; with little information per observation it underfits. Does
more data change its verdict?

In [ ]:
# ── 5.3 BIC with four times more data ──
_, O_big = sample_hmm_many(160, 100, pi, trans, emit, np.random.default_rng(SEED + 53))     # 16,000 SIMULATED rolls
for n_states in (1, 2, 3):
    p_n, a_n, b_n, h_n = fit_best(O_big, n_states, seeds=[700 + 10 * n_states + r for r in range(2)])
    d = (n_states - 1) + n_states * (n_states - 1) + n_states * 5
    print(f"n = {n_states}: train log-lik {h_n[-1]:10.1f}   BIC {-2 * h_n[-1] + d * np.log(O_big.size):10.1f}")

🔍 **What just happened?** With 16,000 rolls the 2-state model gains about 160 nats over one die, while BIC's extra
penalty has grown only to $8\times\log(16000)/2 \approx 39$ nats (the gain grows with $N$, the penalty only like
$\log N$) — now BIC picks $n = 2$ (lowest value above). Lessons:

* the "right" number of states depends on **how much data you have**, not only on the truth: with few observations, a
  simpler model can genuinely predict better;
* held-out likelihood and BIC can disagree; held-out evaluation answers the predictive question directly but needs data
  set aside (and, for sequences, the split must respect the sequence structure — whole sequences, not random positions);
* ⚠️ for HMMs, "$N$" in BIC is ambiguous (rolls? sequences?) because neighbouring observations are dependent; the number
  of observations is the usual convention.

✏️ **Your turn.** Simulate data from a **three-state** model (add a die C that strongly favours 3s, and a transition
matrix with 0.85 on the diagonal) and repeat 5.1. Does held-out likelihood find 3? How many rolls does BIC need?

### 📘 Lecture notes — how many hidden states?

**The problem.** Baum–Welch takes the number of states as an input. More states means a more complex model, which
always fits the training data at least as well — and overfits. Choosing $n$ is a **model-selection problem**:

- **cross-validation / held-out likelihood:** fit on training sequences, compare $\log P(O_{\text{test}}\mid\hat\theta)$
  on held-out sequences (split by whole sequences, not by positions);
- **an information criterion** such as BIC $=-2\log P(O\mid\hat\theta)+d\log N$ with $d=(n-1)+n(n-1)+n(m-1)$ free
  parameters (lower is better);
- **nonparametric Bayesian models** (e.g. the infinite HMM), which let the data decide how many states to use.

**What the simulation shows.** For data from the two-dice model, held-out likelihood picks $n=2$; BIC picks $n=1$
with 4,000 rolls (the 2-state gain of ≈27 nats is smaller than the extra penalty of ≈33 nats) but $n=2$ with 16,000
rolls (gain ≈160 nats vs penalty ≈39 nats). The "right" $n$ depends on how much data you have.

> 🎓 **Exam tip.** Never choose $n$ by training likelihood. Be able to count the free parameters $d = (n-1)+n(n-1)+n(m-1)$ and to say why BIC's penalty $d\log N$ grows more slowly than the likelihood gain of a genuinely better model.

---
## 6 · Dynamic Bayesian networks: the HMM family

> 📍 **Where we are.** An HMM is a Bayesian network repeated over time with *one* hidden variable per time step.
> Biology often needs more: several interacting hidden processes, continuous measurements, inputs such as treatments,
> or realistic segment lengths.

A **dynamic Bayesian network** (DBN) is a Bayesian network for modelling a dynamic process: the state at time $t$ is
represented by a *set* of random variables $\mathbf Q_t$, and the model specifies $P(\mathbf Q_t\mid\mathbf Q_{t-1})$ with a
small graph. An HMM is the special case that represents the state with a single random variable, $\mathbf Q_t =
\{q_t\}$. Let us draw the classic variants, two time slices each (orange = observed); the next cell then *unrolls*
three of them over several time steps.

In [ ]:
# ── 6.1 Five members of the family as DBNs ──
designs = {
    "HMM": ([("q₁", "q₂"), ("q₁", "y₁"), ("q₂", "y₂")],
            {"q₁": (.25, .75), "q₂": (.75, .75), "y₁": (.25, .2), "y₂": (.75, .2)}),
    "factorial HMM (D = 2)": ([("a₁", "a₂"), ("b₁", "b₂"), ("a₁", "y₁"), ("b₁", "y₁"), ("a₂", "y₂"), ("b₂", "y₂")],
            {"a₁": (.12, .85), "a₂": (.62, .85), "b₁": (.38, .55), "b₂": (.88, .55), "y₁": (.25, .15), "y₂": (.75, .15)}),
    "mixture-of-Gaussians HMM": ([("q₁", "q₂"), ("q₁", "m₁"), ("q₂", "m₂"), ("m₁", "y₁"), ("m₂", "y₂"), ("q₁", "y₁"), ("q₂", "y₂")],
            {"q₁": (.3, .85), "q₂": (.8, .85), "m₁": (.12, .5), "m₂": (.62, .5), "y₁": (.3, .15), "y₂": (.8, .15)}),
    "autoregressive HMM": ([("q₁", "q₂"), ("q₁", "y₁"), ("q₂", "y₂"), ("y₁", "y₂")],
            {"q₁": (.25, .75), "q₂": (.75, .75), "y₁": (.25, .2), "y₂": (.75, .2)}),
    "input–output HMM": ([("u₁", "q₁"), ("u₂", "q₂"), ("q₁", "q₂"), ("q₁", "y₁"), ("q₂", "y₂"), ("u₁", "y₁"), ("u₂", "y₂")],
            {"u₁": (.12, .88), "u₂": (.62, .88), "q₁": (.3, .52), "q₂": (.8, .52), "y₁": (.12, .12), "y₂": (.62, .12)}),
}
fig, axes = plt.subplots(1, 5, figsize=(12, 2.9))
for ax, (title, (edges, pos)) in zip(axes, designs.items()):
    observed = {v for v in pos if v[0] in "yu"}
    draw_graph(edges, pos, ax=ax, highlight=observed, radius=0.075, title=title)
plt.tight_layout(); plt.show()

In [ ]:
# ── 6.1b Unrolled over time: a factorial HMM, a semi-Markov HMM and a hierarchical HMM ──
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1.15, 1.15, 1]})
# (a) factorial HMM: D = 3 chains (a, b, c) of hidden states, unrolled over T = 4 steps
pos_f, edges_f = {}, []
for t in range(1, 5):
    x = 0.2 + 0.25 * (t - 1)
    for name, dx, y in [("a", 0.0, 0.9), ("b", -0.07, 0.62), ("c", -0.14, 0.34)]:   # staggered so arrows clear nodes
        pos_f[f"{name}{t}"] = (x + dx, y)
        if t > 1:
            edges_f.append((f"{name}{t - 1}", f"{name}{t}"))
        edges_f.append((f"{name}{t}", f"y{t}"))
    pos_f[f"y{t}"] = (x + 0.06, 0.06)
draw_graph(edges_f, pos_f, ax=axes[0], highlight={f"y{t}" for t in range(1, 5)}, radius=0.05,
           title="factorial HMM: D = 3 chains, unrolled over 4 steps\n(K³ joint states, but only 3 small transition tables)")
# (b) semi-Markov HMM: each visit lasts L steps and emits L observations
seg = {"q1": ["y1", "y2", "y3"], "q2": ["y4", "y5"], "q3": ["y6", "y7", "y8"]}
pos_s = {"q1": (0.13, 0.8), "q2": (0.5, 0.8), "q3": (0.87, 0.8)}
xs = iter([0.02, 0.13, 0.24, 0.44, 0.56, 0.76, 0.87, 0.98])
edges_s = [("q1", "q2"), ("q2", "q3")]
for q, ys in seg.items():
    for yname in ys:
        pos_s[yname] = (next(xs), 0.2)
        edges_s.append((q, yname))
draw_graph(edges_s, pos_s, ax=axes[1], highlight={y for ys in seg.values() for y in ys}, radius=0.05,
           title="semi-Markov HMM: explicit durations\nP(L = l | q) = Poisson(λ_q); segments of length 3, 2, 3")
for q, ys in seg.items():
    axes[1].text(np.mean([pos_s[y][0] for y in ys]), 0.0, f"L = {len(ys)}", ha="center", fontsize=8, color="0.3")
# (c) hierarchical HMM: each top-level state runs its own sub-HMM that emits a sub-sequence
pos_h = {"T1": (0.25, 0.88), "T2": (0.75, 0.88),
         "s1": (0.08, 0.5), "s2": (0.25, 0.5), "s3": (0.42, 0.5), "s4": (0.63, 0.5), "s5": (0.87, 0.5),
         **{f"y{k}": (x, 0.12) for k, x in zip(range(1, 6), [0.08, 0.25, 0.42, 0.63, 0.87])}}
edges_h = [("T1", "T2"), ("T1", "s1"), ("T1", "s2"), ("T1", "s3"), ("T2", "s4"), ("T2", "s5"),
           ("s1", "s2"), ("s2", "s3"), ("s4", "s5")] + [(f"s{k}", f"y{k}") for k in range(1, 6)]
draw_graph(edges_h, pos_h, ax=axes[2], highlight={f"y{k}" for k in range(1, 6)}, radius=0.05,
           title="hierarchical HMM: each state emits an HMM\n(top states T call sub-HMMs s → observations y)")
plt.tight_layout(); plt.show()

How to read them (the first figure shows two time slices of each model; the second unrolls three of them over
time), with a biological use for each:

* **Factorial HMM** (Ghahramani & Jordan 1997): a DBN with $D$ chains, each with $K$ states, jointly producing each
  observation — e.g. the copy-number states of two parental chromosomes, both affecting a measured allele signal.
* **Mixture-of-Gaussians HMM**: each state emits a continuous value from a mixture of Gaussians (hidden component $m_t$) —
  e.g. log-ratios of array or sequencing read depth in copy-number segmentation, or ion-channel currents.
* **Autoregressive HMM**: the observation also depends on the previous observation — e.g. gene expression or
  protein levels over time, where today's value is yesterday's value plus a state-dependent change.
* **Input–output HMM**: an observed input $u_t$ (a drug dose, a stimulus, a cell-cycle signal) influences the
  transitions and emissions.
* **Hierarchical HMM**: each state can emit another HMM that generates a sub-sequence (e.g. "gene" → exons,
  introns → codons).

**The price of richness.** A factorial HMM with $D$ chains of $K$ states is equivalent to an ordinary HMM with $K^D$ joint
states. Let us count what that means.

In [ ]:
# ── 6.2 Factorial HMMs: structure keeps the parameter count small ──
rows = []
for K in (2, 3, 4):
    for D in (1, 2, 3, 5, 8):
        joint_states = K ** D
        rows.append((K, D, joint_states, joint_states * (joint_states - 1), D * K * (K - 1), joint_states ** 2))
print(pd.DataFrame(rows, columns=["K", "D chains", "joint states K^D", "transition params (flat HMM)",
                                  "transition params (factorial)", "forward cost per step (flat)"]).to_string(index=False))

🔍 **What just happened?** With 8 binary chains a flat HMM would have 256 joint states and 65,280 transition
parameters; the factorial structure needs 16. *Conditional independence buys tractability* — again. (Exact inference is
still expensive: the chains become dependent once the shared observation is seen — "explaining away", Module 16 — so
the forward algorithm on the flat joint state costs $(K^D)^2$ per step. Exploiting the factorial structure — updating
one chain at a time — reduces this to $O(D\cdot K^{D+1})$ (Ghahramani & Jordan 1997), still exponential in $D$, so
approximate inference is used for large $D$.)

### 6.1 Semi-Markov HMMs: how long does a state last?

In an HMM the only way to stay in state $s_i$ is to take the self-transition again and again, each time with
probability $a_{ii}$. The duration $L$ of a visit is therefore **geometric**:

$$P(L = l) = a_{ii}^{\,l-1}\,(1-a_{ii}),\qquad l = 1, 2, \dots,\qquad \mathbb E[L] = \frac{1}{1-a_{ii}}.$$

*In words:* stay $l-1$ times, then leave once. The most likely duration is always $l = 1$, and long visits become
exponentially rare. For the dice, $a_{AA} = 0.8$ gives mean 5 rolls — but a visit of exactly 1 roll is the most common.

Real biological segments often have a *typical* length: a typical internal exon of a human gene is roughly 100–200 bp
long, and very short exons are rare. A **semi-Markov HMM** relaxes the Markov constraint to allow staying in the
current state for an explicit duration of time: it draws a duration $L_t$ for each visit,
e.g. $P(L_t = l\mid Q_t) = \text{Poisson}(\lambda_{Q_t})$, then emits $l$ observations from that state:
$P(y_{t-l+1:t}\mid Q_t, L_t = l) = \prod_{i=t-l+1}^{t} P(y_i\mid Q_t)$ — *in words:* the $l$ observations of the
segment are emitted independently by the same state. (Both the index range and the product run over the $l$
positions $t-l+1,\dots,t$ of the segment that ends at $t$.) GENSCAN-style gene finders
use such explicit length distributions (Burge & Karlin 1997).
⚠️ A Poisson can return 0, which is not a valid duration; we use $L = 1 + \text{Poisson}(\lambda - 1)$ so that $L \ge 1$
and the mean is $\lambda$.

In [ ]:
# ── 6.3 Geometric (HMM) vs explicit (semi-Markov) durations, both with mean 5 ──
rng = np.random.default_rng(SEED + 6)
Q_long, _ = sample_hmm_many(1, 20_000, pi, trans, emit, rng)          # one long SIMULATED dice chain
q = Q_long[0]
change = np.flatnonzero(np.diff(q) != 0) + 1
bounds = np.r_[0, change, len(q)]
runs_A = np.diff(bounds)[q[bounds[:-1]] == 0]                          # lengths of the visits to die A
L_semi = 1 + rng.poisson(5 - 1, size=len(runs_A))                     # semi-Markov: 1 + Poisson(4), mean 5

ls = np.arange(1, 26)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(ls - 0.2, [np.mean(runs_A == l) for l in ls], 0.4, color="tab:blue", label="HMM (simulated runs of die A)")
ax.plot(ls, 0.8 ** (ls - 1) * 0.2, "k.-", lw=1, label=r"geometric $0.8^{l-1}\cdot 0.2$")
ax.bar(ls + 0.2, [np.mean(L_semi == l) for l in ls], 0.4, color="tab:green", label="semi-Markov: 1 + Poisson(4)")
ax.set_xlabel("duration of a visit l (rolls)"); ax.set_ylabel("fraction of visits")
ax.set_title("same mean duration (5), very different shapes"); ax.legend()
plt.tight_layout(); plt.show()
print(f"HMM runs of A: n = {len(runs_A)}, mean {runs_A.mean():.2f}, P(L = 1) = {np.mean(runs_A == 1):.3f}, "
      f"P(L ≥ 12) = {np.mean(runs_A >= 12):.3f}")
print(f"semi-Markov : mean {L_semi.mean():.2f}, P(L = 1) = {np.mean(L_semi == 1):.3f}, P(L ≥ 12) = {np.mean(L_semi >= 12):.3f}")

🔍 **What just happened?** The simulated visits to die A follow the geometric formula (black dots) closely: about 20 %
last a single roll, and the distribution has a long tail. The semi-Markov durations with the *same mean* are peaked
around 4–5, with almost no visits of length 1 and far fewer very long ones. If your biology has a typical segment length,
an ordinary HMM is the wrong model for durations — use explicit durations, or the classic trick of replacing one state by
a chain of identical sub-states (which turns the geometric into a negative-binomial distribution).

✏️ **Your turn.** Implement the trick: an "A" made of 3 sub-states A₁ → A₂ → A₃ (each with self-loop probability
$p$, and A₃ exits to B), with emissions identical to die A. Simulate visit lengths for $p = 0.4$ — what is their mean and
mode? (Mean $3/(1-p) = 5$; the mode is no longer 1.)

### 📘 Lecture notes — dynamic Bayesian networks: the HMM family

**Dynamic Bayesian networks (DBNs)** are Bayesian networks for modelling dynamic processes. An HMM represents the
state with a single random variable, $P(Q_t\mid Q_{t-1})$; a DBN represents it with a *set* of random variables
$\mathbf Q_t$, $P(\mathbf Q_t\mid\mathbf Q_{t-1})$. The HMM is a special case.

**Variants.**
- *Factorial HMM:* $D$ chains, each with $K$ states, jointly emit each observation — equivalent to a flat HMM with
  $K^D$ states, but with only $D\cdot K(K-1)$ transition parameters instead of $K^D(K^D-1)$.
- *Mixture-of-Gaussians HMM:* each state emits a continuous value from a Gaussian mixture.
- *Autoregressive HMM:* the observation also depends on the previous observation.
- *Input–output HMM:* an observed input influences transitions and emissions.
- *Semi-Markov HMM:* relaxes the Markov constraint to allow staying in the current state for an explicit duration
  $L_t$, $P(L_t=l\mid Q_t)=\text{Poisson}(\lambda_{Q_t})$, with the $l$ observations of the segment emitted independently:
  $P(y_{t-l+1:t}\mid Q_t,L_t=l)=\prod_{i=t-l+1}^{t}P(y_i\mid Q_t)$.
- *Hierarchical HMM:* each state can emit another HMM that generates a sub-sequence.

**Why durations matter.** In an ordinary HMM a visit to $s_i$ lasts $L$ steps with the geometric distribution

$$P(L=l)=a_{ii}^{\,l-1}(1-a_{ii}),\qquad\mathbb E[L]=\frac{1}{1-a_{ii}},$$

*In words:* stay $l-1$ times, then leave — the most likely duration is always 1. Exons have a typical length, so gene
finders use explicit durations (semi-Markov) or chains of sub-states (a negative-binomial duration). A plain Poisson
can return 0, so in code we use $L=1+\text{Poisson}(\lambda-1)$.

> 🎓 **Exam tip.** Be able to say which distribution of durations an ordinary HMM implies (geometric, mean $1/(1-a_{ii})$, mode 1) and why that matters for genes. For a factorial HMM, compute the size of the equivalent flat state space.

---
## 7 · HMMs in sequence biology

> 📍 **Where we are.** The lecture topic closes with biological HMMs: transcription-factor motifs, profile HMMs for sequence
> alignment, and phylo-HMMs for comparative genomics. All three are learned with the tools of Sections 1–4.

### 7.1 From motifs to profiles

A transcription factor binds short DNA words — **DNA motifs**. When the promoters of the same gene are aligned across
related species, the binding sites stand out as short blocks that are conserved, while the DNA around them has
diverged: mutations inside a site destroy binding and are removed by selection. A classic example aligns yeast
promoters of five *Saccharomyces* species, in which the sites of the TATA-binding protein (TBP) and of the regulators
GAL4 and MIG1 are boxed. The next cell simulates an alignment of that kind.

In [ ]:
# ── 7.0 Picture: a SIMULATED promoter alignment of five yeast species with conserved binding sites ──
rng_motif = np.random.default_rng(SEED + 70)
L_al = 90
SITES = {"GAL4": (6, "CGGAGTACTGTCCTCCG"), "MIG1": (42, "GCGGGG"), "TBP": (68, "TATAAA")}   # start column, consensus
ancestor = rng_motif.choice(list("ACGT"), L_al)
in_site = np.zeros(L_al, bool)
for start, motif in SITES.values():
    ancestor[start:start + len(motif)] = list(motif)
    in_site[start:start + len(motif)] = True
SPECIES = ["S. cerevisiae", "S. paradoxus", "S. mikatae", "S. kudriavzevii", "S. bayanus"]
promoters = []
for sp in SPECIES:
    seq = ancestor.copy()
    mutated = rng_motif.random(L_al) < np.where(in_site, 0.03, 0.35)   # sites mutate rarely (purifying selection)
    seq[mutated] = [rng_motif.choice([b for b in "ACGT" if b != x]) for x in seq[mutated]]
    promoters.append("".join(seq))
identical = np.array([len(set(col)) == 1 for col in zip(*promoters)])
print(f"columns identical in all 5 species: inside binding sites {identical[in_site].mean():.0%}, "
      f"elsewhere {identical[~in_site].mean():.0%}")

BASE_COL = {"A": "tab:green", "C": "tab:blue", "G": "tab:orange", "T": "tab:red"}
fig, ax = plt.subplots(figsize=(12, 2.6))
for r, (sp, seq) in enumerate(zip(SPECIES, promoters)):
    ax.text(-1.5, -r, sp, ha="right", va="center", fontsize=8, style="italic")
    for c, ch in enumerate(seq):
        ax.text(c, -r, ch, ha="center", va="center", fontsize=7, family="monospace", color=BASE_COL[ch])
for c in np.flatnonzero(identical):
    ax.text(c, -len(SPECIES), "*", ha="center", va="center", fontsize=8, color="0.3")
for name, (start, motif) in SITES.items():
    ax.add_patch(plt.Rectangle((start - 0.5, -len(SPECIES) + 0.55), len(motif), len(SPECIES) - 0.1, fill=False,
                               ec="tab:red", lw=1.5))
    ax.text(start + len(motif) / 2 - 0.5, 0.75, name, ha="center", fontsize=9, color="tab:red", weight="bold")
ax.set_xlim(-14, L_al); ax.set_ylim(-len(SPECIES) - 0.6, 1.3); ax.axis("off")
ax.set_title("SIMULATED promoter alignment of five yeast species (* = identical column): binding sites stay conserved",
             fontsize=9)
plt.show()

🔍 **What just happened?** Outside the binding sites, the simulated promoters have diverged so much that only a minority
of columns are still identical in all five species; inside the GAL4, MIG1 and TBP sites almost every column is (the
printout gives both fractions). Two models of this section exploit exactly this pattern: a **motif model** describes
what a site looks like, position by position, and a **phylo-HMM** (Section 7.2) finds conserved stretches without
knowing the motif in advance.

A binding motif is usually summarised by **position-specific** base probabilities — one emission table per position.
Written as an HMM, that is a left-to-right chain of states without self-loops: state 1 emits the first base, state 2
the second, and so on. Real families of sequences also contain **insertions** and **deletions**; a **profile HMM**
(introduced for protein families by Krogh et al. 1994; Krogh 1998 is a tutorial) adds states for them. Its parameters are learned from a multiple alignment **by counting** — Section 1,
with the alignment providing the hidden path of each sequence.

A classic small example (an HMM for a DNA motif alignment, from Krogh's tutorial) uses five aligned DNA sequences.
Columns 4–6 are mostly gaps, so the model treats them as an **insertion** between the third and fourth "match"
positions:

```text
A C A - - - A T G
T C A A C T A T C
A C A C - - A G C
A G A - - - A T C
A C C G - - A T C
```

In [ ]:
# ── 7.1 Build the profile HMM of the five-sequence alignment by counting ──
alignment = ["ACA---ATG", "TCAACTATC", "ACAC--AGC", "AGA---ATC", "ACCG--ATC"]
BASES = "ACGT"
gap_frac = [np.mean([s[c] == "-" for s in alignment]) for c in range(9)]
print("fraction of gaps per column:", np.round(gap_frac, 1))
print("a '≤ 50 % gaps → match' rule would choose columns", [c + 1 for c in range(9) if gap_frac[c] <= 0.5])
match_cols, insert_cols = [0, 1, 2, 6, 7, 8], [3, 4, 5]                 # the example's choice (0-based)
STATE_NAMES = ["M1", "M2", "M3", "I", "M4", "M5", "M6"]                 # match states and one insert state

emit_counts = np.zeros((7, 4)); trans_counts = np.zeros((7, 7))
for s in alignment:
    path, letters = [], []
    for k, c in enumerate(match_cols[:3]):
        path.append(k); letters.append(s[c])
    for c in insert_cols:
        if s[c] != "-":
            path.append(3); letters.append(s[c])
    for k, c in enumerate(match_cols[3:]):
        path.append(4 + k); letters.append(s[c])
    for st, ch in zip(path, letters):
        emit_counts[st, BASES.index(ch)] += 1                           # emission counts (Section 1!)
    for a, b in zip(path[:-1], path[1:]):
        trans_counts[a, b] += 1                                         # transition counts along the path
emit_prof = emit_counts / emit_counts.sum(1, keepdims=True)
row_tot = trans_counts.sum(1, keepdims=True)
trans_prof = np.divide(trans_counts, row_tot, out=np.zeros_like(trans_counts), where=row_tot > 0)
print("\nemission probabilities (rows = states):")
print(pd.DataFrame(emit_prof, index=STATE_NAMES, columns=list(BASES)).round(2))
print("\nnon-zero transition probabilities:")
print(", ".join(f"{STATE_NAMES[a]}→{STATE_NAMES[b]} {trans_prof[a, b]:.1f}" for a, b in zip(*np.nonzero(trans_prof))))

🔍 **What just happened?** Counting gives every number of the model (drawn in the next cell): match state 1 emits A with 0.8 and T
with 0.2; the insert state emits A .2, C .4, G .2, T .2 (the five inserted letters A, C, T, C, G); three of five sequences
enter the insert state after M3 (0.6) and two go straight on (0.4); inside the insert state, two of the five inserted
letters are followed by another insertion (self-loop 0.4) and three by M4 (0.6). Note that the column choice is itself
a modelling decision: a common "≤ 50 % gaps" rule would have made column 4 a match state; the example (following
Krogh's tutorial) keeps all three gappy columns as one insertion. The picture below is the classic way to draw such a
model: each state shows its emission histogram, and each arrow is as thick as its transition probability.

In [ ]:
# ── 7.1b Picture: the profile HMM — states with emission histograms, arrows as thick as their probability ──
from matplotlib.patches import FancyArrowPatch, Polygon, Rectangle

fig, ax = plt.subplots(figsize=(11, 3.9))
XPOS = {"M1": 0.0, "M2": 1.4, "M3": 2.8, "M4": 5.6, "M5": 7.0, "M6": 8.4}
BW, BH = 0.95, 1.15                                         # size of a match-state box


def draw_histogram(x0, y0, probs, w, h):
    """Four horizontal bars (A, C, G, T) inside a w × h box whose lower-left corner is (x0, y0)."""
    for r, (base, p) in enumerate(zip(BASES, probs)):
        yy = y0 + h - (r + 0.5) * h / 4
        ax.text(x0 + 0.05, yy, base, fontsize=8, va="center", family="monospace")
        ax.add_patch(Rectangle((x0 + 0.24, yy - 0.09), (w - 0.32) * p, 0.18, fc="tab:blue" if p > 0 else "none"))


def arrow(p0, p1, prob, rad=0.0, label_xy=None):
    """Transition arrow whose width grows with the probability; the probability is written next to it."""
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=12, lw=0.6 + 4 * prob, color="0.3",
                                 connectionstyle=f"arc3,rad={rad}"))
    lx, ly = label_xy or ((p0[0] + p1[0]) / 2, (p0[1] + p1[1]) / 2 + 0.12)
    ax.text(lx, ly, f"{prob:.1f}", fontsize=8, ha="center", color="0.15")


for k, name in enumerate(STATE_NAMES):
    if name == "I":
        continue
    x0 = XPOS[name]
    ax.add_patch(Rectangle((x0, 0), BW, BH, fc="white", ec="0.2", lw=1.2))
    draw_histogram(x0, 0, emit_prof[k], BW, BH)
    ax.text(x0 + BW / 2, -0.22, name, ha="center", va="top", fontsize=9)
cx, cy, rr = 4.7, 2.15, 0.85                                # the insert state: a diamond above the gap
ax.add_patch(Polygon([(cx, cy + rr), (cx + rr, cy), (cx, cy - rr), (cx - rr, cy)], closed=True, fc="white", ec="0.2", lw=1.2))
draw_histogram(cx - 0.42, cy - 0.42, emit_prof[STATE_NAMES.index("I")], 0.84, 0.84)
ax.text(cx + rr + 0.1, cy + 0.35, "I (insert)", fontsize=9)
T_ = lambda a, b: trans_prof[STATE_NAMES.index(a), STATE_NAMES.index(b)]
for a, b in [("M1", "M2"), ("M2", "M3"), ("M4", "M5"), ("M5", "M6")]:
    arrow((XPOS[a] + BW, BH / 2), (XPOS[b], BH / 2), T_(a, b))
arrow((XPOS["M3"] + BW, BH / 2), (XPOS["M4"], BH / 2), T_("M3", "M4"))                       # skip the insertion
arrow((XPOS["M3"] + 0.75 * BW, BH), (cx - 0.5, cy - 0.38), T_("M3", "I"), label_xy=(3.95, 1.75))
arrow((cx + 0.5, cy - 0.38), (XPOS["M4"] + 0.25 * BW, BH), T_("I", "M4"), label_xy=(5.5, 1.75))
arrow((cx - 0.3, cy + rr - 0.15), (cx + 0.3, cy + rr - 0.15), T_("I", "I"), rad=-2.2, label_xy=(cx, cy + rr + 0.62))
ax.set_xlim(-0.3, 9.6); ax.set_ylim(-0.55, 3.75); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("profile HMM learned from the 5-sequence alignment: boxes = match states, diamond = insert state,\n"
             "bars = emission probabilities of A, C, G, T; arrow width ∝ transition probability", fontsize=9)
plt.show()

Now use the model. The probability of a sequence is the sum over all paths through the profile that emit it — the
**forward algorithm** of Module 19, with the extra rule that the path must start in M1 and end in M6.

In [ ]:
# ── 7.2 Score sequences with the profile HMM (forward algorithm, start in M1, end in M6) ──
def profile_log_prob(seq: str, trans_p, emit_p) -> float:
    """log P(seq | profile HMM): scaled forward pass starting in M1; only paths ending in M6 count."""
    obs = [BASES.index(ch) for ch in seq]
    start = np.zeros(7); start[0] = 1.0
    a = start * emit_p[:, obs[0]]
    log_c = 0.0
    for o in obs[1:]:
        if a.sum() == 0:
            return -np.inf
        log_c += np.log(a.sum()); a = a / a.sum()                      # scaling, as in Module 19
        a = (a @ trans_p) * emit_p[:, o]
    return log_c + np.log(a[6]) if a[6] > 0 else -np.inf


by_hand = 0.8 * 1 * 0.8 * 1 * 0.8 * 0.6 * 0.4 * 0.6 * 1 * 1 * 0.8 * 1 * 0.8
print(f"P(ACACATC): forward {np.exp(profile_log_prob('ACACATC', trans_prof, emit_prof)):.4f} | "
      f"by hand along its only path M1 M2 M3 I M4 M5 M6: {by_hand:.4f}")
tests = alignment + ["ACACATC", "ACAATC", "TGCTAGG", "GCAATC"]
for s in tests:
    seq = s.replace("-", "")
    lp = profile_log_prob(seq, trans_prof, emit_prof)
    print(f"{seq:10s} log P = {lp:8.3f}   log-odds vs random DNA (0.25 per base) = {lp - len(seq) * np.log(0.25):8.3f}")

🔍 **What just happened?** The forward algorithm gives $P(\text{ACACATC}) = 0.0472$, the same as multiplying the
probabilities along its unique path by hand. The **log-odds** score compares the profile with a background model of
random DNA: positive means "looks like a family member". The training sequences and the consensus-like ACAATC score
well; TGCTAGG — every letter allowed by the profile but rarely typical — scores below zero; and GCAATC gets
$-\infty$ because no training sequence had a G in the first column, so $\hat b_{M1}(G) = 0$. Exactly the zero-count
problem of Section 1 — and the same cure.

In [ ]:
# ── 7.3 Pseudo-counts for the profile's emissions, and a sequence-logo-like picture ──
emit_prof_pc = (emit_counts + 0.5) / (emit_counts + 0.5).sum(1, keepdims=True)
for s in ["GCAATC", "TGCTAGG", "ACAATC"]:
    print(f"{s:8s} log-odds with pseudo-count 0.5: {profile_log_prob(s, trans_prof, emit_prof_pc) - len(s) * np.log(0.25):7.3f}")

fig, ax = plt.subplots(figsize=(7, 3))
bottom = np.zeros(7)
for b, color in zip(range(4), ["tab:green", "tab:blue", "tab:orange", "tab:red"]):
    ax.bar(STATE_NAMES, emit_prof[:, b], bottom=bottom, color=color, label=BASES[b], edgecolor="white")
    for k in range(7):
        if emit_prof[k, b] >= 0.2:
            ax.text(k, bottom[k] + emit_prof[k, b] / 2, BASES[b], ha="center", va="center", color="white", weight="bold")
    bottom += emit_prof[:, b]
ax.set_ylabel("emission probability"); ax.set_title("profile HMM: base composition of each state")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With a pseudo-count of 0.5 the unseen G becomes merely unlikely, and the scores order the
sequences sensibly. The stacked bars are a crude **sequence logo**: conserved positions (M4 is always A) are a single
colour; variable ones are mixtures. Real profile HMMs (HMMER, Pfam) have a match, an insert *and* a delete state for every
column, learn them from large alignments with carefully designed priors, and use Viterbi through the profile to align a
new sequence to the family — the alignment *is* the decoded hidden path.

### 7.2 Phylo-HMMs: conserved elements across species

In an alignment of a microRNA locus across vertebrates (human, chimpanzee, mouse, cow, chicken, …) some columns are
identical in every species (purifying selection — the sequence matters), others vary. A **phylo-HMM** (Siepel *et al.*
2005) parses aligned sequences into two classes, labelling each alignment column as **c** (conserved) or **n**
(non-conserved). The hidden chain is an ordinary two-state HMM with transition probabilities $P(c\to n) = \mu$ and
$P(n\to c) = \nu$; the begin probabilities $\nu/(\mu+\nu)$ and $\mu/(\mu+\nu)$ are the
chain's **stationary distribution** (the long-run fraction of columns in each state). The *emission* of a state is the
probability of a whole alignment column under a **phylogenetic tree** — with short branches for $c$ (little change) and
longer branches for $n$.

**Toy version (SIMULATED).** Three species on a star tree: the ancestral base is uniform, and along each branch it changes
to one of the other three bases with probability $p_c = 0.05$ (conserved) or $p_n = 0.4$ (non-conserved). A column
$(x_1,x_2,x_3)$ then has probability

$$P(x_1,x_2,x_3\mid \text{state}) = \sum_{a\in\{A,C,G,T\}} \tfrac14 \prod_{s=1}^{3} P(x_s\mid a),\qquad
P(x\mid a) = \begin{cases}1-p & x = a\\ p/3 & x\neq a.\end{cases}$$

*In words:* sum over the unknown ancestral base; given the ancestor, the three species mutate independently. There are
$4^3 = 64$ possible columns, so the emission "table" has 64 symbols — and all our HMM code works unchanged.

In [ ]:
# ── 7.4 Emission table of the toy phylo-HMM, and the stationary begin probabilities ──
def column_probs(p_change: float) -> np.ndarray:
    """P(column) for all 64 three-species columns under a star tree with per-branch change probability p_change."""
    P_x_given_a = np.full((4, 4), p_change / 3); np.fill_diagonal(P_x_given_a, 1 - p_change)
    probs = np.zeros(64)
    for idx, (x1, x2, x3) in enumerate(itertools.product(range(4), repeat=3)):
        probs[idx] = sum(0.25 * P_x_given_a[a, x1] * P_x_given_a[a, x2] * P_x_given_a[a, x3] for a in range(4))
    return probs


emit_phylo = np.vstack([column_probs(0.05), column_probs(0.4)])     # state 0 = c (conserved), 1 = n
mu, nu = 0.02, 0.01                                                 # P(c → n), P(n → c)
trans_phylo = np.array([[1 - mu, mu], [nu, 1 - nu]])
pi_phylo = np.array([nu, mu]) / (mu + nu)                           # begin probabilities ν/(μ+ν), μ/(μ+ν)
identical = [i for i, col in enumerate(itertools.product(range(4), repeat=3)) if len(set(col)) == 1]
print("each emission row sums to 1:", np.allclose(emit_phylo.sum(1), 1))
print(f"P(all three species identical): conserved {emit_phylo[0, identical].sum():.3f}, "
      f"non-conserved {emit_phylo[1, identical].sum():.3f}")
print("begin probabilities ν/(μ+ν), μ/(μ+ν) =", np.round(pi_phylo, 3), "| stationary? π·a = π:",
      np.allclose(pi_phylo @ trans_phylo, pi_phylo))

In [ ]:
# ── 7.4b Picture: the phylo-HMM — a two-state chain whose emissions are phylogenetic trees ──
from matplotlib.patches import Circle, FancyArrowPatch

fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 1.25]})
ax = axes[0]
nodes = {"begin": (0.5, 0.86), "c": (0.27, 0.4), "n": (0.73, 0.4)}
for name, (x, y) in nodes.items():
    fc = {"begin": "white", "c": "0.35", "n": "0.85"}[name]
    ax.add_patch(Circle((x, y), 0.075 if name != "begin" else 0.065, fc=fc, ec="0.2", lw=1.2))
    ax.text(x, y, name, ha="center", va="center", fontsize=11 if name != "begin" else 8,
            color="white" if name == "c" else "black", weight="bold")


def link(p0, p1, label, rad=0.0, dx=0.0, dy=0.0):
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=12, lw=1.2, color="0.3",
                                 connectionstyle=f"arc3,rad={rad}"))
    ax.text((p0[0] + p1[0]) / 2 + dx, (p0[1] + p1[1]) / 2 + dy, label, fontsize=9, ha="center")


link((0.45, 0.81), (0.32, 0.47), "ν/(μ+ν)", dx=-0.1)
link((0.55, 0.81), (0.68, 0.47), "μ/(μ+ν)", dx=0.1)
link((0.34, 0.43), (0.66, 0.43), "μ", rad=-0.3, dy=0.09)
link((0.66, 0.37), (0.34, 0.37), "ν", rad=-0.3, dy=-0.11)
link((0.22, 0.46), (0.22, 0.34), "1 − μ", rad=1.9, dx=-0.17)
link((0.78, 0.34), (0.78, 0.46), "1 − ν", rad=1.9, dx=0.17)
for (x0, y0), length, label in [((0.13, 0.12), 0.05, "ψ_c: short branches"), ((0.75, 0.12), 0.13, "ψ_n: long branches")]:
    for dxl in (-1, 0, 1):                                   # a star tree with three leaves
        ax.plot([x0, x0 + dxl * length], [y0 + 0.1, y0 + 0.1 - length * 0.9], color="0.3", lw=1.2)
    ax.text(x0 + 0.02, y0 - 0.06, label, fontsize=8, ha="center")
ax.set_xlim(0, 1); ax.set_ylim(-0.02, 1); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("states c (conserved) / n (non-conserved): transitions μ, ν;\nemission probabilities = a phylogenetic tree per state", fontsize=9)

pos_p, edges_p = {}, []                                     # graphical model: hidden chain, one alignment column per step
for t in range(1, 4):
    x = 0.17 + 0.33 * (t - 1)
    pos_p[f"q{t}"] = (x, 0.88)
    pos_p[f"a{t}"] = (x, 0.55)                               # unobserved ancestral base of the column
    edges_p.append((f"q{t}", f"a{t}"))
    if t > 1:
        edges_p.append((f"q{t - 1}", f"q{t}"))
    for s, dxs in zip((1, 2, 3), (-0.1, 0.0, 0.1)):
        pos_p[f"x{t}{s}"] = (x + dxs, 0.15)                  # base of species s in column t (observed)
        edges_p.append((f"a{t}", f"x{t}{s}"))
draw_graph(edges_p, pos_p, ax=axes[1], highlight={v for v in pos_p if v[0] == "x"}, radius=0.045,
           title="phylo-HMM as a graphical model: hidden state q_t → tree for column t\n(a_t = ancestral base, x_ts = base of species s; orange = observed)")
plt.tight_layout(); plt.show()

In [ ]:
# ── 7.5 Simulate an alignment and decode conserved elements ──
rng = np.random.default_rng(SEED + 7)
Q_ph, O_ph = sample_hmm_many(1, 3000, pi_phylo, trans_phylo, emit_phylo, rng)
S_ph, _, ll_ph = forward_backward(O_ph, pi_phylo, trans_phylo, emit_phylo)
p_cons = S_ph[0, :, 0]
pred_c = (p_cons > 0.5).astype(int)
truth_c = (Q_ph[0] == 0).astype(int)
same = np.isin(O_ph[0], identical)
print(f"SIMULATED alignment: 3,000 columns, {truth_c.mean():.1%} conserved")
print(f"decoding accuracy (posterior > 0.5): {accuracy(truth_c, pred_c):.3f}")
print(f"baseline 'conserved iff the three bases are identical': {accuracy(truth_c, same.astype(int)):.3f}")

lo, hi = 0, 800
x = np.arange(lo, hi)
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.fill_between(x, 0, truth_c[lo:hi], step="mid", color="tab:green", alpha=0.25, label="truly conserved (hidden)")
ax.plot(x, p_cons[lo:hi], color="tab:blue", lw=1.2, label="P(conserved | alignment)")
ax.plot(x[~same[lo:hi]], np.full((~same[lo:hi]).sum(), -0.06), "|", color="0.4", ms=5, label="column not identical")
ax.set_ylim(-0.12, 1.05); ax.set_xlabel("alignment column"); ax.set_ylabel("probability")
ax.set_title("SIMULATED 3-species alignment: a toy phylo-HMM finds conserved elements")
ax.legend(loc="upper center", bbox_to_anchor=(0.36, 0.98), fontsize=7); plt.tight_layout(); plt.show()

🔍 **What just happened?** A single column is weak evidence: 22 % of non-conserved columns are identical by chance and
14 % of conserved columns are not (cell 7.4), so the column-by-column baseline is right only about 80 % of the time. The phylo-HMM pools evidence along
the alignment and recovers the conserved stretches with high accuracy (printed above): long runs of identical columns
switch the posterior to "conserved", and a dense sprinkle of differences switches it back. Real phylo-HMMs (phastCons)
use realistic trees with many species, substitution models with branch lengths, and parameters $\mu$, $\nu$ and the tree
scaling **learned** by EM-type algorithms from the alignment itself — the same E-step/M-step logic as Section 4.

✏️ **Your turn.** Make conservation weaker ($p_c = 0.15$) and re-run 7.4–7.5. How does the accuracy change? Then
re-estimate $\mu$ and $\nu$ with `baum_welch` on `O_ph`, holding the emissions fixed (hint: copy `baum_welch` and skip the
emission update). Do you recover 0.02 and 0.01?

### 📘 Lecture notes — HMMs in sequence biology: motifs, profile HMMs, phylo-HMMs

**DNA motifs.** Transcription factors bind short DNA motifs (e.g. the TBP, GAL4 and MIG1 sites conserved across yeast
promoters). A motif is a set of position-specific base distributions — an HMM that moves left to right, one state
per position.

**A profile HMM for sequence alignment.** Match states (one per conserved column, each with its own base histogram),
insert states (extra letters) and, in full models, delete states. Learned from a multiple alignment by **counting**
along each sequence's path. Example alignment `ACA---ATG`, `TCAACTATC`, `ACAC--AGC`, `AGA---ATC`, `ACCG--ATC`
(columns 4–6 = an insertion): M1 emits A .8, T .2; the insert state emits A .2, C .4, G .2, T .2; transitions
M3 → I .6, M3 → M4 .4, I → I .4, I → M4 .6, all other match-to-match transitions 1.0. Scoring: the forward algorithm
through the profile, $P(\text{ACACATC})=.8\cdot1\cdot.8\cdot1\cdot.8\cdot.6\cdot.4\cdot.6\cdot1\cdot1\cdot.8\cdot1\cdot.8=0.0472$;
compare with random DNA by the log-odds score; use pseudo-counts for unseen letters.

**Phylo-HMM.** Parse aligned sequences from several species into two classes: hidden states **c** (conserved
region) and **n** (non-conserved region); transition probabilities $\mu=P(c\to n)$, $\nu=P(n\to c)$; begin
probabilities $\nu/(\mu+\nu)$ and $\mu/(\mu+\nu)$ (the stationary distribution); emission probabilities $\psi_c$,
$\psi_n$ given by a **phylogenetic tree** for each state (short branches for c, long for n). Observed: one alignment
column (one base per species) per position; unobserved: the state, and the ancestral bases inside the tree.

**What you should know.** Why HMMs; inference in HMMs — the forward–backward algorithm, various inference tasks, the
Viterbi algorithm (Module 19); learning in HMMs — the EM algorithm with inference as a subroutine (this module).

> 🎓 **Exam tip.** For a profile HMM, be able to estimate emission and transition probabilities from a small alignment by counting, and to compute the probability of a sequence along its path. For a phylo-HMM, state what is hidden (conserved or not), what is observed (alignment columns) and what plays the role of the emission table (a tree model).

---
## 📌 Key takeaways

What you should know about learning in HMMs — "the EM algorithm with inference as a subroutine" — and the
generalisations that close the lecture topic:

- **States observed → MLE by counting.** The log-likelihood splits into initial, transition and emission parts; each row
  is a categorical distribution, so $\hat\pi_i$, $\hat a_{ij} = N_{ij}/\sum_{j'}N_{ij'}$ and $\hat b_i(\sigma) = M_i(\sigma)/\sum M_i$
  are count ratios (worked example: $\pi_A = 2/3$, $a_{AB} = 4/9$, $b_A(5) = 1/5$). Pseudo-counts prevent zero probabilities.
- **States hidden → EM (Baum–Welch).** E-step: expected counts $S_t(i) = \alpha_t(i)\beta_t(i)/P(O)$ and
  $S_t(i,j) = \alpha_t(i)a_{ij}b_j(o_{t+1})\beta_{t+1}(j)/P(O)$ from forward–backward — *inference as a subroutine*. M-step:
  the same count ratios with expected counts.
- **EM's behaviour.** The likelihood never decreases; the answer depends on the start (local optima, plateaus) and is
  identified only up to relabelling the states; restarts and sensible initialisations matter; hidden-state models need
  much more data than labelled ones.
- **Latent variables ⇒ EM**, the course's third recurring idea: an HMM is a mixture model whose labels are chained over time.
- **Number of states = model selection.** Training likelihood always prefers more states; use held-out likelihood or BIC
  (which can disagree, and whose verdict depends on the amount of data).
- **DBNs generalise HMMs**: factorial, mixture-of-Gaussians, autoregressive, input–output, semi-Markov (explicit, non-geometric
  durations) and hierarchical HMMs — structure keeps the parameter count manageable.
- **Biological HMMs** — gene finders, profile HMMs for protein/DNA families, phylo-HMMs for conservation — are all learned with
  the counting/EM machinery of this notebook.

---
## ✅ Checkpoint

1. From the three labelled example sequences, compute $\hat a_{BB}$ and $\hat b_B(6)$ by hand.
   <details><summary>Answer</summary>Departures from B: BA 3 + BB 6 = 9, so $\hat a_{BB} = 6/9 = 2/3$. Die B is used 11 times and rolls a 6 three times (once in each sequence), so $\hat b_B(6) = 3/11 \approx 0.27$.</details>
2. Why does the HMM log-likelihood with observed states split into three separate maximisation problems?
   <details><summary>Answer</summary>Because the log of the product $\pi\prod a\prod b$ is a sum of three groups of terms — initial, transition, emission — that involve disjoint sets of parameters, and each group is a sum of count × log(parameter) for categorical distributions.</details>
3. What are $S_t(i)$ and $S_t(i,j)$, and how are they computed?
   <details><summary>Answer</summary>$S_t(i) = P(q_t = s_i\mid O) = \alpha_t(i)\beta_t(i)/\sum_j\alpha_t(j)\beta_t(j)$; $S_t(i,j) = P(q_t=s_i, q_{t+1}=s_j\mid O) = \alpha_t(i)a_{ij}b_j(o_{t+1})\beta_{t+1}(j)/P(O)$. Their sums over $t$ (and sequences) are the expected state and transition counts used by the M-step.</details>
4. Your Baum–Welch log-likelihood goes *down* between two iterations by 3.2 nats. What do you conclude?
   <details><summary>Answer</summary>There is a bug (or a numerical problem): EM can never decrease the observed-data likelihood. Typical culprits: forgetting the emission or the $1/c_{t+1}$ factor in $S_t(i,j)$, normalising transition counts over columns instead of rows, or computing the log-likelihood with different parameters than those used in the E-step.</details>
5. Two runs of Baum–Welch on the same data return emission tables that are each other's row-swapped versions, with identical log-likelihoods. Which run is right?
   <details><summary>Answer</summary>Both: this is label switching. Permuting the state names (rows/columns of $a$, rows of $b$, entries of $\pi$) leaves $P(O)$ unchanged, so the states are identified only up to relabelling.</details>
6. With 4,000 rolls BIC chose 1 state although the data came from 2 dice; with 16,000 it chose 2. Explain.
   <details><summary>Answer</summary>The 2-state model's log-likelihood gain grows roughly linearly with $N$, but BIC's extra penalty $\tfrac12 \Delta d\log N$ ($\Delta d = 8$) grows only logarithmically. With 4,000 rolls the gain (≈27 nats) is smaller than the penalty (≈33 nats); with 16,000 rolls the gain (≈160 nats) far exceeds the penalty (≈39 nats).</details>
7. An HMM state has self-transition probability 0.95. What is the distribution of its visit lengths, and what is wrong with using it for exons?
   <details><summary>Answer</summary>Geometric: $P(L=l) = 0.95^{l-1}\times0.05$, mean 20, with mode 1 and an exponential tail. Exon lengths have a typical size with few very short exons, so a semi-Markov (explicit-duration) model or a chain of sub-states fits better.</details>
8. Code reading: in `m_step`, what does `np.einsum('kti,ktm->im', S, np.eye(m)[O])` compute?
   <details><summary>Answer</summary>For every state $i$ and symbol $\sigma$, the sum of the soft labels $S^k_t(i)$ over all sequences $k$ and positions $t$ where $o^k_t = \sigma$ — the expected number of times state $i$ emitted $\sigma$, the $B_i(\sigma)$ of the M-step.</details>

## 🧠 Questions & discussion

1. Section 2's rule-based labels were 'mostly right' yet produced absurd parameters. When, in a real biology project, might hard labels from a simple rule still be a reasonable starting point for EM?
2. Baum–Welch found parameters with a *higher* training log-likelihood than the true ones. Is that a problem? What would you look at to decide whether a learned HMM is trustworthy?
3. If the hidden states of a learned HMM are identified only up to relabelling, how do biologists decide that state 3 of ChromHMM 'is' an active promoter? What external evidence is needed?
4. Held-out likelihood picked 2 states while BIC picked 1. Which would you report in a paper about the number of chromatin states in a cell type, and how would you phrase the uncertainty?
5. A profile HMM trained on 5 sequences gives log-odds $-\infty$ to a true family member with one unusual residue. Discuss the trade-off in choosing the pseudo-count.

## 🚀 Bonus — going further

- **Viterbi training (hard EM).** Replace the E-step by Viterbi decoding and count the decoded path (the HMM analogue of k-means vs Gaussian mixtures in [Module 13](13_mixture_models_em.ipynb)). Compare speed and final parameters with Baum–Welch on the simulated dice.
- **MAP Baum–Welch.** Add Dirichlet pseudo-counts to the expected counts in `m_step` (Module 05). Show that this prevents emission probabilities from collapsing to 0 when training on very little data (e.g. 200 rolls).
- **Bayesian learning by sampling.** Instead of EM, alternate (i) sampling a state path with forward-filtering backward-sampling and (ii) sampling the parameters from their Dirichlet posteriors given the counts — a Gibbs sampler ([Module 21](21_bayesian_learning_mcmc.ipynb)).
- **A full profile HMM.** Give every column a match, insert and delete state, train from a Pfam seed alignment of your choice, and align new sequences with Viterbi.
- For the MLE/MAP background see the repository notebook [Fundamentals of learning under uncertainty](../../FundamentalsLearningUncertainty/); the sister course [BiologyToResearch](../../../../cryoem/tutorial/BiologyToResearch/) applies EM to classify cryo-ET subtomograms.
- **From profile HMMs to protein language models.** [Module 37](37_protein_sequences_language_models.ipynb) models protein families with Potts models (pairwise couplings between columns, which a profile HMM ignores) and protein language models; [Module 43](43_llms_ai_agents.ipynb) follows next-symbol prediction from Markov chains to large language models.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 20 · Hidden Markov models II — learning, Baum–Welch and beyond
Input: K sequences of observations (dice rolls / alignment columns), with or without their hidden states (SIMULATED,
       plus the 3 labelled example sequences and the 5-sequence alignment)
Output: estimated π, a, b; log-likelihood history; chosen number of states; decoded segments
Method: counting MLE (+ pseudo-counts) when states are observed; Baum–Welch EM (scaled forward–backward E-step,
        count-ratio M-step) when hidden; model selection by held-out log-likelihood and BIC
Key equation: S_t(i,j) = α_t(i) a_ij b_j(o_{t+1}) β_{t+1}(j) / P(O);  a_ij = Σ_t S_t(i,j) / Σ_t Σ_j' S_t(i,j')
Key parameters: number of states n, initialisation, tolerance / max iterations, number of restarts, pseudo-count α
What changed when I modified the amount of data / the separation of the dice / the initialisation: …
One failure mode: EM stalls on a plateau or local optimum; zero counts make unseen events impossible; BIC underfits
                  with little data
Connection to the 02-620 topics (which idea of the lecture this illustrates): MLE by counting; EM/Baum–Welch with
                             inference as a subroutine; number of states; DBNs and semi-Markov durations;
                             profile HMMs and phylo-HMMs
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists* (course design S. Kim; instructors M. Xu and M. Zhang) — this module teaches the second part of its lecture on hidden Markov models and dynamic Bayesian networks (learning, dynamic Bayesian networks, biological HMMs).
- Rabiner L. R. (1989) A tutorial on hidden Markov models and selected applications in speech recognition. *Proceedings of the IEEE* 77:257–286.
- Baum L. E., Petrie T., Soules G., Weiss N. (1970) A maximization technique occurring in the statistical analysis of probabilistic functions of Markov chains. *Annals of Mathematical Statistics* 41:164–171.
- Dempster A. P., Laird N. M., Rubin D. B. (1977) Maximum likelihood from incomplete data via the EM algorithm. *Journal of the Royal Statistical Society B* 39:1–38.
- Durbin R., Eddy S. R., Krogh A., Mitchison G. (1998) *Biological Sequence Analysis: Probabilistic Models of Proteins and Nucleic Acids*. Cambridge University Press — Chapters 3 (HMM parameter estimation) and 5 (profile HMMs).
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — Chapter 17 (HMMs, Baum–Welch, semi-Markov models) and Chapter 11 (EM).
- Krogh A. (1998) An introduction to hidden Markov models for biological sequences. In Salzberg S. L., Searls D. B., Kasif S. (eds) *Computational Methods in Molecular Biology*, pp. 45–63. Elsevier.
- Krogh A., Brown M., Mian I. S., Sjölander K., Haussler D. (1994) Hidden Markov models in computational biology: applications to protein modeling. *Journal of Molecular Biology* 235:1501–1531.
- Eddy S. R. (1998) Profile hidden Markov models. *Bioinformatics* 14:755–763.
- Siepel A. *et al.* (2005) Evolutionarily conserved elements in vertebrate, insect, worm, and yeast genomes. *Genome Research* 15:1034–1050.
- Ghahramani Z., Jordan M. I. (1997) Factorial hidden Markov models. *Machine Learning* 29:245–273.
- Burge C., Karlin S. (1997) Prediction of complete gene structures in human genomic DNA. *Journal of Molecular Biology* 268:78–94.
- Schwarz G. (1978) Estimating the dimension of a model. *Annals of Statistics* 6:461–464.
- Ernst J., Kellis M. (2012) ChromHMM: automating chromatin-state discovery and characterization. *Nature Methods* 9:215–216.

## ➡️ Next

EM finds one best set of parameters. [Module 21 · Bayesian learning and MCMC](21_bayesian_learning_mcmc.ipynb) asks for the whole *distribution* of plausible parameters, and shows how Gibbs sampling — the sampling sibling of EM — explores it.